# PURGE CACHED MODELS

In [ ]:
import os
import shutil
import stat


model_paths = [
    "span_prediction_42.pt",
    "./xlmr-qa-final",
    "week39_lr_model.pkl",
    "week39_lr_vectorizer.pkl",
    "week39_lstm_model.pt",
    "week39_vocab.json",
    "/xlmr-qa-final",
    "./roberta-qa-translated-final"
    ]

def purge(path):
    if not path:
        return
    if os.path.islink(path) or os.path.isfile(path):
        os.remove(path)
    elif os.path.isdir(path):
        shutil.rmtree(path)


for path in model_paths:
    purge(path)
    print(f"Removed model on path: {path}")

# Generic Utilities

## Generic Imports

In [ ]:
import numpy as np
import pandas as pd
import os

## Torch and CUDA setup

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Dataset operations

### Imports

In [ ]:
from datasets import load_dataset, DatasetDict
import nltk
nltk.download("punkt_tab")

### Functions

In [ ]:
def get_dataset() -> DatasetDict:
    """
    Prepare the dataset for training and evaluation.
    """
    ds = load_dataset("coastalcph/tydi_xor_rc")
    languages = {"ar", "ko", "te"}
    ds = ds.filter(lambda language: language["lang"] in languages)
    return ds

def nltk_tokenize_dataset(dataset) -> DatasetDict:
    def tokenize_text(text):
        return {
            "question_tokens": nltk.word_tokenize(text["question"]),
            "context_tokens": nltk.word_tokenize(text["context"])
        }

    return dataset.map(tokenize_text)

## Translation to English

### Imports and preparation

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
TRANSLATION_MODEL_NAME = "facebook/nllb-200-distilled-600M"

# Check if the model is already downloaded, if not, download it
if not os.path.exists(f"./{TRANSLATION_MODEL_NAME}"):
    print(f"Downloading {TRANSLATION_MODEL_NAME}...")
    translation_tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL_NAME)
    translation_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATION_MODEL_NAME)
    translation_model.save_pretrained(f"./{TRANSLATION_MODEL_NAME}")
    translation_tokenizer.save_pretrained(f"./{TRANSLATION_MODEL_NAME}")
else:
    print(f"Loading {TRANSLATION_MODEL_NAME} from local directory...")  
    translation_tokenizer = AutoTokenizer.from_pretrained(f"./{TRANSLATION_MODEL_NAME}")
    translation_model = AutoModelForSeq2SeqLM.from_pretrained(f"./{TRANSLATION_MODEL_NAME}")

# Load model to device (CPU or GPU)
translation_model = translation_model.to(device)
translation_model.eval()

print(f"Loaded {TRANSLATION_MODEL_NAME} on {device}")

LANG_CODES = {
    "ar": "arb_Arab",
    "ko": "kor_Hang",
    "te": "tel_Telu",
}

### Functions

In [ ]:
def translate_to_english(text, lang):
    translation_tokenizer.src_lang = LANG_CODES[lang]

    inputs = translation_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
    ).to(device)

    with torch.inference_mode():
        outputs = translation_model.generate(
            **inputs,
            forced_bos_token_id=translation_tokenizer.convert_tokens_to_ids(
                "eng_Latn"
            )
        )

    return translation_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
    )

def translate_batch_to_english(questions, lang, batch_size=16):
    """Translate a batch of questions"""
    if not questions:
        return []

    translation_tokenizer.src_lang = LANG_CODES[lang]
    translations = []

    for start in range(0, len(questions), batch_size):
        batch = questions[start:start + batch_size]

        inputs = translation_tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(device)

        with torch.inference_mode():
            outputs = translation_model.generate(
                **inputs,
                forced_bos_token_id=translation_tokenizer.convert_tokens_to_ids(
                    "eng_Latn"
                ),
                num_beams=1,
            )

        batch_translations = translation_tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True,
        )

        translations.extend(batch_translations)

    return translations

# W36 - Data and rule-based baselines

## Dataset Preparation

In [ ]:
ds = get_dataset()
ds = nltk_tokenize_dataset(ds)

## Dataset statistics

### Part 1 - \# of examples, answerable/unanswerable proportions, median, IQ-range of tokenized question and context lengths, missing values, and exact duplicate Q/C pairs

#### Imports

In [ ]:
import pandas as pd
import numpy as np

#### Functions

In [ ]:
# Compute stats
languages = {"ar", "ko", "te"}
stats = []
for lang in languages:
    lang_ds = ds.filter(lambda example: example["lang"] == lang)
    for split in ["train", "validation"]:
        split_ds = lang_ds[split]
        num_examples = len(split_ds)

        answerability_ratio = sum(1 for example in split_ds if example["answerable"]) / num_examples

        median_length_question = np.median([len(example["question_tokens"]) for example in split_ds])
        median_length_context = np.median([len(example["context_tokens"]) for example in split_ds])

        q1_question = np.percentile([len(example["question_tokens"]) for example in split_ds], 25)
        q3_question = np.percentile([len(example["question_tokens"]) for example in split_ds], 75)
        iq_range_question = q3_question - q1_question
        q1_context = np.percentile([len(example["context_tokens"]) for example in split_ds], 25)
        q3_context = np.percentile([len(example["context_tokens"]) for example in split_ds], 75)
        iq_range_context = q3_context - q1_context

        missing = 0
        for example in split_ds:
            for field in example.keys():
                if example[field] is None:
                    missing += 1


        seen = set()
        duplicates = 0
        for example in split_ds:
            pair = (example["question"], example["context"])
            if pair in seen:
                duplicates += 1
            else:
                seen.add(pair)

        stats.append({
            "lang": lang,
            "split": split,
            "num_examples": num_examples,
            "answerability_ratio": answerability_ratio,
            "median_length_question": median_length_question,
            "median_length_context": median_length_context,
            "iq_range_question": iq_range_question,
            "iq_range_context": iq_range_context,
            "missing": missing,
            "duplicates": duplicates
        })

# Compute totals for entire dataset
stats.append({
    "lang": "all",
    "split": "all",
    "num_examples": sum(stat["num_examples"] for stat in stats),
    "answerability_ratio": sum(stat["answerability_ratio"] * stat["num_examples"] for stat in stats) / sum(stat["num_examples"] for stat in stats),
    "median_length_question": np.median([stat["median_length_question"] for stat in stats]),
    "median_length_context": np.median([stat["median_length_context"] for stat in stats]),
    "iq_range_question": np.median([stat["iq_range_question"] for stat in stats]),
    "iq_range_context": np.median([stat["iq_range_context"] for stat in stats]),
    "missing": sum(stat["missing"] for stat in stats),
    "duplicates": sum(stat["duplicates"] for stat in stats)
})

df = pd.DataFrame(stats)

#### Evaluation

In [ ]:
# Export to latex
part_1_tex = df.to_latex(header=True, index=False, float_format="%.3f")
print(part_1_tex)

### Part 2 - Five most common question tokens and counts per language, english translation

#### Functions

In [ ]:
async def translate_token(token, lang):
    try:
        return translate_to_english(token, lang)
    except Exception as e:
        return f"Error: {e}"


common_tokens = {}

for lang in languages:
    token_counts = {}
    lang_ds = ds.filter(lambda example: example["lang"] == lang)

    for example in lang_ds["train"]:
        for token in example["question_tokens"]:
            token_counts[token] = token_counts.get(token, 0) + 1

    common_tokens[lang] = sorted(
        token_counts.items(),
        key=lambda x: x[1],
        reverse=True,
    )[:5]


translated_tokens = {}

for lang, tokens in common_tokens.items():
    translated_tokens[lang] = []

    for token, count in tokens:
        translation = await translate_token(token, lang)
        translated_tokens[lang].append((token, translation, count))

#### Evaluation

In [ ]:
for token in translated_tokens:
    print(f"Language: {token}")
    for original, translation, count in translated_tokens[token]:
        print(f"Original: {original}, Translation: {translation}, Count: {count}")
    print()

### Part 3 - Verify that `answer` equals substring from `answer_start`

In [ ]:
correct_count = 0
incorrect_count = 0
for lang in languages:
    for split in ["train", "validation"]:
        split_ds = ds.filter(lambda example: example["lang"] == lang)[split]
        for example in split_ds:
            if not example["answerable"]:
                continue
            substring = example["context"][example["answer_start"]:]
            if substring.startswith(example["answer"]):
                correct_count += 1
            else:
                incorrect_count += 1

print(f"Correctly aligned: {correct_count}")
print(f"Incorrectly aligned: {incorrect_count}")

## Answerability baselines

### Majority-class baseline from training split

#### Imports

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

#### Functions

In [ ]:
def evaluate_majority_baseline(train_ds, eval_ds):
    answerable_count = sum(train_ds["answerable"])
    unanswerable_count = len(train_ds) - answerable_count
    majority_class = answerable_count >= unanswerable_count

    y_true = list(eval_ds["answerable"])
    y_pred = [majority_class] * len(y_true)

    print("Majority class:", majority_class)
    print("Evaluation examples:", len(y_true))
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print(
        "Macro-F1:",
        f1_score(y_true, y_pred, average="macro", zero_division=0),
    )
    print(
        classification_report(
            y_true,
            y_pred,
            labels=[True, False],
            target_names=["Answerable", "Unanswerable"],
            zero_division=0,
        )
    )

    return majority_class

#### Evaluation

In [ ]:
for lang in ["ar", "ko", "te"]:
    lang_ds = ds.filter(lambda x: x["lang"] == lang)

    print(f"\n{lang}")
    evaluate_majority_baseline(
        lang_ds["train"],
        lang_ds["validation"],
    )

### Deterministic Rule-based Classifier

#### Imports

In [ ]:
from nltk.corpus import stopwords
from sklearn.metrics import accuracy_score, f1_score
import re
nltk.download("stopwords", quiet=True)

english_stopwords = set(stopwords.words("english"))
translation_cache = {}

#### Functions

In [ ]:
def content_tokens(text):
    tokens = re.findall(r"\b[a-z]+\b", text.lower())
    return {
        token for token in tokens
        if token not in english_stopwords
    }


def predict_answerable(
    question,
    context,
    lang,
    threshold=0.3,
    translated_question=None,
):
    if translated_question is None:
        translated_question = translation_cache.get(
            (lang, question), question
        )

    question_words = content_tokens(translated_question)
    context_words = content_tokens(context)

    if not question_words:
        return False

    overlap_ratio = len(question_words & context_words) / len(question_words)
    return overlap_ratio >= threshold


async def evaluate_rule_baseline(eval_ds, threshold=0.3, batch_size=16):
    langs = sorted(set(eval_ds["lang"]))
    translation_errors = []

    for lang in langs:
        questions = list(dict.fromkeys(
            example["question"]
            for example in eval_ds
            if example["lang"] == lang
        ))

        missing = [
            question for question in questions
            if (lang, question) not in translation_cache
        ]

        for start in range(0, len(missing), batch_size):
            batch = missing[start:start + batch_size]

            try:
                translated = translate_batch_to_english(
                    batch,
                    lang,
                    batch_size=batch_size,
                )

                for question, translated_text in zip(batch, translated):
                    if (
                        translated_text == question
                        and question.strip()
                        and lang != "en"
                    ):
                        raise RuntimeError(
                            f"Translation returned original text: {lang}"
                        )

                    translation_cache[(lang, question)] = translated_text

            except Exception:
                for question in batch:
                    try:
                        translated_text = await translate_to_english(
                            question, lang
                        )

                        if (
                            translated_text == question
                            and question.strip()
                            and lang != "en"
                        ):
                            raise RuntimeError(
                                "Translation returned original text"
                            )

                        translation_cache[(lang, question)] = translated_text

                    except Exception as exc:
                        translation_errors.append({
                            "lang": lang,
                            "question": question,
                            "error_type": type(exc).__name__,
                            "error_message": str(exc),
                        })

    rows = []

    for example in eval_ds:
        lang = example["lang"]
        question = example["question"]
        translated_question = translation_cache.get(
            (lang, question), question
        )

        prediction = predict_answerable(
            question=question,
            context=example["context"],
            lang=lang,
            threshold=threshold,
            translated_question=translated_question,
        )

        rows.append({
            "lang": lang,
            "question": question,
            "translated_question": translated_question,
            "context": example["context"],
            "answerable": bool(example["answerable"]),
            "predicted": bool(prediction),
            "is_correct": (
                bool(prediction) == bool(example["answerable"])
            ),
        })

    predictions_df = pd.DataFrame(rows)
    errors_df = pd.DataFrame(
        translation_errors,
        columns=[
            "lang",
            "question",
            "error_type",
            "error_message",
        ],
    )

    return predictions_df, errors_df


def summarize_predictions(pred_df, split_name, threshold):
    y_true = pred_df["answerable"].astype(bool).tolist()
    y_pred = pred_df["predicted"].astype(bool).tolist()

    return pd.DataFrame([{
        "split": split_name,
        "n_examples": len(pred_df),
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "support_answerable": int(pred_df["answerable"].sum()),
        "support_unanswerable": int((~pred_df["answerable"]).sum()),
        "pred_answerable": int(pred_df["predicted"].sum()),
        "pred_unanswerable": int((~pred_df["predicted"]).sum()),
    }])

#### Evaluation

In [ ]:
translation_cache.clear()
threshold = 0.3
max_concurrency = 20

lang_results = {}
metric_rows = []

for lang in ["ar", "ko", "te"]:
    lang_ds = ds.filter(lambda example: example["lang"] == lang)

    pred_df, errors_df = await evaluate_rule_baseline(
        lang_ds["validation"],
        threshold=threshold,
        batch_size=max_concurrency,
    )

    lang_results[lang] = pred_df
    lang_results[f"errors_{lang}"] = errors_df

    metric_rows.append(
        summarize_predictions(
            pred_df,
            split_name=f"validation_{lang}",
            threshold=threshold,
        )
    )

overall_pred_df, overall_errors_df = await evaluate_rule_baseline(
    ds["validation"],
    threshold=threshold,
    batch_size=max_concurrency,
)

lang_results["overall"] = overall_pred_df
lang_results["errors_overall"] = overall_errors_df

metric_rows.append(
    summarize_predictions(
        overall_pred_df,
        split_name="validation_overall",
        threshold=threshold,
    )
)

metrics_df = pd.concat(metric_rows, ignore_index=True)
display(metrics_df)

for lang in ["ar", "ko", "te"]:
    errors_df = lang_results[f"errors_{lang}"]

    if not errors_df.empty:
        print(f"\nTranslation errors for {lang}:")
        display(errors_df)

In [ ]:
# test inspect
for split_name, pred_df in lang_results.items():
    print(f"\n{split_name}")
    display(pred_df.head())

print("\nMetrics summary:")
display(metrics_df)

In [ ]:
metrics_latex = metrics_df.to_latex(index=False, float_format="%.4f")

metrics_latex = metrics_latex.replace("_", "\\_")
print("\nMetrics LaTeX table:")
print(metrics_latex)

# W37 - Structured Span Prediction

## Imports

In [ ]:
import random
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm import tqdm
from statistics import mean, stdev
from collections import defaultdict
from transformers import BertModel
import torch.nn as nn

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## Dataset Preparation



In [ ]:
ds = get_dataset()

## BIO features preparation

In [ ]:
# O, B, I, and -100 (ignore) labels for BIO tagging

def prepare_bio_features(examples, tokenizer, max_length=512, doc_stride=0):
    tokenized_inputs = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        # Retain offset for character spans
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_inputs.pop("overflow_to_sample_mapping")
    
    all_labels = []
    
    for i in range(len(tokenized_inputs["input_ids"])):
        sample_idx = sample_mapping[i]
        
        is_answerable = examples["answerable"][sample_idx]
        ans_start = examples["answer_start"][sample_idx]
        
        if not is_answerable or ans_start == -1:
            is_answerable = False
            ans_end = -1
        else:
            ans_text = examples["answer"][sample_idx]
            ans_end = ans_start + len(ans_text)
            
        sequence_ids = tokenized_inputs.sequence_ids(i)
        word_ids = tokenized_inputs.word_ids(i)
        offsets = tokenized_inputs["offset_mapping"][i]
        
        labels = [-100] * len(offsets)
        
        previous_word_idx = None
        is_first_answer_token = True
        
        for j, (seq_id, word_id, offset) in enumerate(zip(sequence_ids, word_ids, offsets)):
            if seq_id != 1 or word_id is None:
                labels[j] = -100
                continue
                
            if word_id == previous_word_idx:
                labels[j] = -100
                continue
                
            previous_word_idx = word_id
            token_start, token_end = offset
            
            if is_answerable:
                if token_start >= ans_start and token_end <= ans_end:
                    if is_first_answer_token:
                        labels[j] = 1 # B
                        is_first_answer_token = False
                    else:
                        labels[j] = 2 # I
                elif token_start < ans_end and token_end > ans_start:
                    if is_first_answer_token:
                        labels[j] = 1
                        is_first_answer_token = False
                    else:
                        labels[j] = 2
                else:
                    labels[j] = 0 # O
            else:
                # else unanswerable
                labels[j] = 0 # O
                
        all_labels.append(labels)
        
    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

### Check output

In [ ]:
def run_automatic_checks():
    tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-multilingual-cased")
    label_map = {0: "O", 1: "B", 2: "I", -100: "-100"}
    
    test_examples = {
        "question": [
            "What colour socks is Søren wearing?",
            "What colour socks is Søren wearing?",
            "What was his name again?",
            "What colour socks is Søren wearing?"
        ],
        "context": [
            "Søren's socks are red.",
            "Søren's socks are red-blue (purple)",
            "Søren, and he is wearing red socks.",
            "The weather is nice today."
        ],
        "answerable": [True, True, True, False],
        "answer_start": [18, 18, 0, -1],
        "answer": ["red", "red-blue (purple)", "Søren", "no"]
    }
    
    features = prepare_bio_features(test_examples, tokenizer)
    
    print("Automatic Checks for BIO Labeller:\n")
    for i in range(len(features["input_ids"])):            
        tokens = tokenizer.convert_ids_to_tokens(features["input_ids"][i])
        labels = features["labels"][i]
        
        context_tokens = []
        context_labels = []
        
        for token, label in zip(tokens, labels):
            if label != -100:
                context_tokens.append(token)
                context_labels.append(label_map[label])
                
        print(f"Example {i+1}:")
        print(f"  Context: {test_examples['context'][i]}")
        print(f"  Tokens : {context_tokens}")
        print(f"  Labels : {context_labels}")
        
        if i == 0: # Punctuation adjacent
            color_idx = context_tokens.index("red")
            assert context_labels[color_idx] == "B", "Failed: Punctuation adjacent"
        if i == 1: # Multi-token:
            assert context_labels[5] == "B" and context_labels[6] == "I", "Failed: Multi-token answer"
        if i == 2: # Answer at char 0
            assert context_labels[0] == "B", "Failed: Answer at char 0"
        elif i == 3: # Unanswerable
            assert all(l == "O" for l in context_labels), "Failed: Unanswerable example"
            
        print()

    print("All automatic checks passed successfully!")

run_automatic_checks()

## Model

In [ ]:
class QuestionConditioned(nn.Module):
    def __init__(self, bert_model_name="google-bert/bert-base-multilingual-cased",
                 num_labels=3, lstm_hidden=128, freeze_embeddings=True):
        super().__init__()
        
        bert = BertModel.from_pretrained(bert_model_name)
        self.embeddings = bert.embeddings
        hidden_size = bert.config.hidden_size
        
        if freeze_embeddings:
            for param in self.embeddings.parameters():
                param.requires_grad = False
        
        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=lstm_hidden,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.1,
        )
        lstm_out_dim = lstm_hidden * 2  # bidirectional
        
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim * 2, lstm_out_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(lstm_out_dim, num_labels),
        )
    
    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        if token_type_ids is None:
            token_type_ids = torch.zeros_like(input_ids)
        
        with torch.no_grad():
            emb = self.embeddings(input_ids, token_type_ids=token_type_ids)
        
        lstm_out, _ = self.lstm(emb)
        
        # Pool question representation
        q_mask = (token_type_ids == 0).float().unsqueeze(-1)
        q_sum = (lstm_out * q_mask).sum(dim=1)
        q_count = q_mask.sum(dim=1).clamp(min=1.0)
        q_pooled = q_sum / q_count
        
        q_expanded = q_pooled.unsqueeze(1).expand_as(lstm_out)
        combined = torch.cat([lstm_out, q_expanded], dim=-1)
        
        logits = self.classifier(combined)
        
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
        
        return {"loss": loss, "logits": logits}


tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-multilingual-cased")

tokenized_datasets = ds.map(
    lambda examples: prepare_bio_features(examples, tokenizer, max_length=512, doc_stride=0),
    batched=True,
    remove_columns=ds["train"].column_names,
    desc="Tokenizing and creating BIO labels"
)

## Training Setup

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = QuestionConditioned().to(device)

train_data = tokenized_datasets["train"].remove_columns(["offset_mapping"])
val_data   = tokenized_datasets["validation"].remove_columns(["offset_mapping"])
train_data.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "labels"])
val_data.set_format("torch",   columns=["input_ids", "attention_mask", "token_type_ids", "labels"])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_data,   batch_size=32, shuffle=False, num_workers=2)

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
num_training_steps = len(train_loader) * 5
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)

def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    for batch in tqdm(loader, desc="train"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs["loss"]
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
    return total_loss / len(loader)



## BIO to span conversion and predictions

In [ ]:
# Convert BIO labels to start,end span
def extract_span_from_bio(predicted_labels, offsets, sequence_ids):
    spans = []
    current_start = None
    
    for label, offset, seq_id in zip(predicted_labels, offsets, sequence_ids):
        if seq_id != 1:
            continue
            
        char_start, char_end = offset
        
        if label == 1:  # B
            if current_start is not None:
                spans.append((current_start, prev_end))
            current_start = char_start
            prev_end = char_end
        elif label == 2:  # I
            if current_start is not None:
                prev_end = char_end 
            else:
                current_start = char_start
                prev_end = char_end
        else:
            if current_start is not None:
                spans.append((current_start, prev_end))
                current_start = None
    
    if current_start is not None:
        spans.append((current_start, prev_end))
    
    if not spans:
        return None 
    
    # Take the first predicted span
    return spans[0]


def predict_span(model, tokenizer, question, context, device="cuda"):
    inputs = tokenizer(
        question, context,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation="only_second",
        max_length=384,
        padding="max_length",
    )
    
    offsets = inputs.pop("offset_mapping")[0].tolist()
    sequence_ids = inputs.sequence_ids(0)
    
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs["logits"]
    
    predictions = torch.argmax(logits, dim=-1)[0].tolist()
    
    span = extract_span_from_bio(predictions, offsets, sequence_ids)
    
    if span is None:
        return ""
    else:
        start, end = span
        return context[start:end]

In [ ]:
val_data_eval = tokenized_datasets["validation"]  # keep offset_mapping

@torch.no_grad()
def predict_bio(model, batch, device):
    batch = {k: v.to(device) for k, v in batch.items() 
             if k in ("input_ids", "attention_mask", "token_type_ids")}
    logits = model(**batch)["logits"]
    return torch.argmax(logits, dim=-1)  # (batch, seq_len)

## Evaluation functions

In [ ]:
def token_metrics(preds, golds):
    """Binary P/R/F1: B/I = positive, O = negative. Skips -100."""
    tp = fp = fn = 0
    for p, g in zip(preds, golds):
        if g == -100:
            continue
        g_pos, p_pos = g in (1, 2), p in (1, 2)
        if   g_pos and p_pos:  tp += 1
        elif not g_pos and p_pos: fp += 1
        elif g_pos and not p_pos: fn += 1
    prec = tp / (tp + fp) if tp + fp > 0 else 0.0
    rec  = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec > 0 else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "prec": prec, "rec": rec, "f1": f1}


def extract_span(preds, offsets, sequence_ids):
    """First contiguous run of B/I among context tokens -> char span."""
    start = end = None
    for p, (cs, ce), sid in zip(preds, offsets, sequence_ids):
        if sid != 1:
            continue
        if p in (1, 2):
            if start is None:
                start = cs
            end = ce
        elif p == 0 and start is not None:
            break  # end of first span
    return (start, end) if start is not None else None

In [ ]:
def gold_for_example(ex, tokenizer, max_length=512):
    """Returns (gold_bio, gold_span_string, offsets, sequence_ids)."""
    answerable = ex["answerable"] and ex["answer_start"] != -1
    gold_span = ex["context"][ex["answer_start"]:ex["answer_start"] + len(ex["answer"])] \
                if answerable else ""

    enc = tokenizer(
        ex["question"], ex["context"],
        truncation="only_second", max_length=max_length,
        padding="max_length", return_offsets_mapping=True,
    )
    offsets   = enc["offset_mapping"]
    seq_ids   = enc.sequence_ids(0)
    word_ids  = enc.word_ids(0)

    gold_bio = [-100] * len(offsets)
    if not answerable:
        for i, (sid, wid) in enumerate(zip(seq_ids, word_ids)):
            if sid == 1 and wid is not None:
                gold_bio[i] = 0
        return gold_bio, gold_span, offsets, seq_ids

    ans_end = ex["answer_start"] + len(ex["answer"])
    first, prev_wid = True, None
    for i, (sid, wid, (cs, ce)) in enumerate(zip(seq_ids, word_ids, offsets)):
        if sid != 1 or wid is None or wid == prev_wid:
            prev_wid = wid
            continue
        prev_wid = wid
        if cs < ans_end and ce > ex["answer_start"]:
            gold_bio[i] = 1 if first else 2
            first = False
        else:
            gold_bio[i] = 0
    return gold_bio, gold_span, offsets, seq_ids

In [ ]:
@torch.no_grad()
def evaluate_model(model, raw_val, tokenizer, device, max_length=512):
    model.eval()
    rows = []
    for ex in tqdm(raw_val, desc="model"):
        gold_bio, gold_span, offsets, seq_ids = gold_for_example(ex, tokenizer, max_length)

        enc = tokenizer(
            ex["question"], ex["context"],
            truncation="only_second", max_length=max_length,
            padding="max_length", return_tensors="pt",
        )
        input_ids      = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)
        token_type_ids = enc.get("token_type_ids", torch.zeros_like(input_ids)).to(device)

        logits = model(input_ids, attention_mask, token_type_ids)["logits"]
        preds  = torch.argmax(logits, dim=-1)[0].tolist()

        tm = token_metrics(preds, gold_bio)
        pred_span_chars = extract_span(preds, offsets, seq_ids)
        pred_span = ex["context"][pred_span_chars[0]:pred_span_chars[1]] if pred_span_chars else ""

        rows.append({
            "lang": ex["lang"],
            "em":   pred_span.strip() == gold_span.strip(),
            **tm,
        })
    return rows

def aggregate(rows, key=None):
    groups = defaultdict(list)
    for r in rows:
        groups[r["lang"] if key == "lang" else "overall"].append(r)
    out = {}
    for name, group in groups.items():
        tp = sum(r["tp"] for r in group)
        fp = sum(r["fp"] for r in group)
        fn = sum(r["fn"] for r in group)
        prec = tp / (tp + fp) if tp + fp > 0 else 0.0
        rec  = tp / (tp + fn) if tp + fn > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if prec + rec > 0 else 0.0
        em   = sum(r["em"] for r in group) / len(group)
        out[name] = {"n": len(group), "P": prec, "R": rec, "F1": f1, "EM": em}
    return out

## Training

In [ ]:
SEEDS = [42, 123, 2024]
all_seed_metrics = []
raw_val = ds["validation"]  

for seed in SEEDS:
    print(f"\n{'='*20} Training with Seed {seed} {'='*20}")
    set_seed(seed)
    
    model = QuestionConditioned().to(device)
    
    optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    num_training_steps = len(train_loader) * 5  # 5 epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * num_training_steps),
        num_training_steps=num_training_steps
    )
    
    for epoch in range(5):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, device)
        print(f"Seed {seed} | Epoch {epoch+1} | train_loss={train_loss:.4f}")
        
    model_path = f"span_prediction_{seed}.pt"
    torch.save(model.state_dict(), model_path)
    print(f"Saved model to {model_path}")
    
    print(f"Evaluating Seed {seed}...")
    rows = evaluate_model(model, raw_val, tokenizer, device)
    
    overall = aggregate(rows)["overall"]
    per_lang = aggregate(rows, key="lang")
    
    seed_dict = {
        "overall": overall,
        "ar": per_lang.get("ar", {}),
        "ko": per_lang.get("ko", {}),
        "te": per_lang.get("te", {})
    }
    all_seed_metrics.append(seed_dict)

### Empty-baseline

In [ ]:
def evaluate_empty_baseline(raw_val, tokenizer, max_length=512):
    rows = []
    for ex in tqdm(raw_val, desc="empty"):
        gold_bio, gold_span, _, _ = gold_for_example(ex, tokenizer, max_length)
        n_pos = sum(1 for g in gold_bio if g in (1, 2))
        rows.append({
            "lang": ex["lang"],
            "em":   gold_span.strip() == "",
            "tp": 0, "fp": 0, "fn": n_pos,
            "prec": 0.0, "rec": 0.0, "f1": 0.0,
        })
    return rows

In [ ]:
def print_table(title, rows):
    overall = aggregate(rows)
    per_lang = aggregate(rows, key="lang")
    print(f"\n=== {title} ===")
    print(f"{'group':<10} {'n':>6} {'P':>7} {'R':>7} {'F1':>7} {'EM':>7}")
    for name in ["overall", "ar", "ko", "te"]:
        r = overall["overall"] if name == "overall" else per_lang.get(name)
        if r is None:
            continue
        print(f"{name:<10} {r['n']:>6} {r['P']:>7.4f} {r['R']:>7.4f} "
              f"{r['F1']:>7.4f} {r['EM']:>7.4f}")

In [ ]:
raw = get_dataset()
val = raw["validation"]

model_rows = evaluate_model(model, val, tokenizer, device)
empty_rows = evaluate_empty_baseline(val, tokenizer)

print_table("Lightweight BiLSTM tagger", model_rows)
print_table("Empty-output baseline",    empty_rows)

## Evaluation

In [ ]:
# Display results for first seed
first_seed = SEEDS[0]
print(f"\nDetailed Results for Seed {first_seed}")
rows_first_seed = evaluate_model(model, raw_val, tokenizer, device)
print_table(f"Lightweight BiLSTM tagger (Seed {first_seed})", rows_first_seed)
print_table(f"Empty-output baseline", empty_rows)

print("\n" + "="*60)
print("Mean ± Standard Deviation Across All Seeds")
print("="*60)
print(f"{'Group':<10} {'Metric':<8} {'Mean':>8} {'Std Dev':>8}")
print("-" * 60)

groups = ["overall", "ar", "ko", "te"]
metrics = ["P", "R", "F1", "EM"]

for group in groups:
    for metric in metrics:
        values = [seed_data[group][metric] for seed_data in all_seed_metrics if group in seed_data and metric in seed_data[group]]
        
        if len(values) > 0:
            m = mean(values)
            s = stdev(values) if len(values) > 1 else 0.0
            print(f"{group:<10} {metric:<8} {m:>8.4f} {s:>8.4f}")
    print("-" * 60) # Separator between groups

# BAD IMPLEMENTATION (NLTK), ONLY FOR COMPARISON - W38 - Language Modelling - Korean

## Dataset Preparation

In [ ]:
ds = get_dataset()
ds = nltk_tokenize_dataset(ds)

## Pavlos - Laplace smoothed unigram for baseline and Laplace smoothed bigram for comparison model

### Imports

In [ ]:
import math
from collections import Counter

### Data preparation and inspection

In [ ]:
ko_train = ds["train"].filter(lambda x: x["lang"] == "ko")
ko_validation = ds["validation"].filter(lambda x: x["lang"] == "ko")

print("Training questions:", len(ko_train))
print("Validation questions:", len(ko_validation))

print("\nExample training question:")
print(ko_train[0]["question"])
print(ko_train[0]["question_tokens"])

print("\nExample validation question:")
print(ko_validation[0]["question"])
print(ko_validation[0]["question_tokens"])

In [ ]:
token_counts = Counter()

for example in ko_train:
    token_counts.update(example["question_tokens"])

vocab = {"<UNK>": 0}

for token in token_counts:
    vocab[token] = len(vocab)

print("Vocabulary size:", len(vocab))
print("Most common tokens:")
print(token_counts.most_common(10))

In [ ]:
unknown_tokens = 0
total_validation_tokens = 0

for example in ko_validation:
    for token in example["question_tokens"]:
        total_validation_tokens += 1
        if token not in vocab:
            unknown_tokens += 1

print("Total validation tokens:", total_validation_tokens)
print("Unknown validation tokens:", unknown_tokens)
print("Unknown-token rate:", unknown_tokens / total_validation_tokens)

In [ ]:
def encode_question(tokens):
    return [
        vocab.get(token, vocab["<UNK>"])
        for token in tokens
    ]

ko_train_encoded = [
    encode_question(example["question_tokens"])
    for example in ko_train
]

ko_validation_encoded = [
    encode_question(example["question_tokens"])
    for example in ko_validation
]

print("Original:")
print(ko_train[0]["question_tokens"])

print("\nEncoded:")
print(ko_train_encoded[0])

In [ ]:
for example, encoded in zip(ko_validation, ko_validation_encoded):
    for token, token_id in zip(example["question_tokens"], encoded):
        if token not in vocab:
            print("Token:", token)
            print("Encoded ID:", token_id)
            print("UNK ID:", vocab["<UNK>"])
            break
    else:
        continue
    break

In [ ]:
START_TOKEN = "<START>"
END_TOKEN = "<END>"

def prepare_lm_tokens(tokens):
    return (
        [START_TOKEN]
        + [token if token in vocab else "<UNK>" for token in tokens]
        + [END_TOKEN]
    )

ko_train_lm = [
    prepare_lm_tokens(example["question_tokens"])
    for example in ko_train
]

ko_validation_lm = [
    prepare_lm_tokens(example["question_tokens"])
    for example in ko_validation
]

print("Original:")
print(ko_train[0]["question_tokens"])

print("\nLM tokens:")
print(ko_train_lm[0])

In [ ]:
vocab_size = len(vocab)
lm_vocab_size = vocab_size + 2 # language-model vocabulary includes the two boundary tokens

print("Original vocabulary size:", vocab_size)
print("LM vocabulary size:", lm_vocab_size)

### Unigram model

In [ ]:
lm_token_counts = Counter()

for tokens in ko_train_lm:
    lm_token_counts.update(tokens)

lm_total_tokens = sum(lm_token_counts.values())

unigram_probs = {}

for token in vocab:
    count = lm_token_counts.get(token, 0)
    unigram_probs[token] = (count + 1) / (lm_total_tokens + lm_vocab_size)

for token in [START_TOKEN, END_TOKEN]:
    count = lm_token_counts.get(token, 0)
    unigram_probs[token] = (count + 1) / (lm_total_tokens + lm_vocab_size)

print("Training LM tokens:", lm_total_tokens)
print("LM vocabulary size:", lm_vocab_size)
print("P(<UNK>):", unigram_probs["<UNK>"])

In [ ]:
log_probability = 0.0
num_tokens = 0

for tokens in ko_validation_lm:
    for token in tokens[1:]:
        log_probability += math.log(unigram_probs[token])
        num_tokens += 1

average_negative_log_probability = -log_probability / num_tokens
unigram_perplexity = math.exp(average_negative_log_probability)

print("Validation unigram tokens:", num_tokens)
print("Unigram validation perplexity:", unigram_perplexity)

In [ ]:
lm_token_counts = Counter()

for tokens in ko_train_lm:
    lm_token_counts.update(tokens)

lm_total_tokens = sum(lm_token_counts.values())

unigram_probs = {}

for token in vocab:
    count = lm_token_counts.get(token, 0)
    unigram_probs[token] = (count + 1) / (lm_total_tokens + lm_vocab_size)

for token in [START_TOKEN, END_TOKEN]:
    count = lm_token_counts.get(token, 0)
    unigram_probs[token] = (count + 1) / (lm_total_tokens + lm_vocab_size)

print("Training LM tokens:", lm_total_tokens)
print("LM vocabulary size:", lm_vocab_size)
print("P(<UNK>):", unigram_probs["<UNK>"])

In [ ]:
log_probability = 0.0
num_tokens = 0

for tokens in ko_validation_lm:
    for token in tokens[1:]:
        log_probability += math.log(unigram_probs[token])
        num_tokens += 1

average_negative_log_probability = -log_probability / num_tokens
unigram_perplexity = math.exp(average_negative_log_probability)

print("Validation unigram tokens:", num_tokens)
print("Unigram validation perplexity:", unigram_perplexity)

### Bigram

In [ ]:
bigram_counts = Counter()
previous_token_counts = Counter()

for tokens in ko_train_lm:
    for i in range(1, len(tokens)):
        previous = tokens[i - 1]
        current = tokens[i]

        bigram_counts[(previous, current)] += 1
        previous_token_counts[previous] += 1

print("Number of unique bigrams:", len(bigram_counts))
print("\nMost common bigrams:")
print(bigram_counts.most_common(10))

In [ ]:
def bigram_probability(previous, current):
    count = bigram_counts.get((previous, current), 0)
    previous_count = previous_token_counts.get(previous, 0)

    return (count + 1) / (previous_count + lm_vocab_size)

In [ ]:
log_probability = 0.0
num_tokens = 0

for tokens in ko_validation_lm:
    for i in range(1, len(tokens)):
        previous = tokens[i - 1]
        current = tokens[i]

        log_probability += math.log(
            bigram_probability(previous, current)
        )
        num_tokens += 1

average_negative_log_probability = -log_probability / num_tokens
bigram_perplexity = math.exp(average_negative_log_probability)

print("Validation bigram tokens:", num_tokens)
print("Bigram validation perplexity:", bigram_perplexity)

### Evaluation

#### Per-Question Unigram Perplexity

In [ ]:
unigram_question_results = []

for question, tokens in zip(ko_validation, ko_validation_lm):
    log_probability = 0.0
    num_tokens = 0

    for token in tokens[1:]:
        log_probability += math.log(unigram_probs[token])
        num_tokens += 1

    question_perplexity = math.exp(-log_probability / num_tokens)

    unigram_question_results.append({
        "question": question["question"],
        "perplexity": question_perplexity,
    })

unigram_question_results.sort(key=lambda x: x["perplexity"])

print("5 lowest-perplexity questions:")
for result in unigram_question_results[:5]:
    print(f"{result['perplexity']:.2f} | {result['question']}")

print("\n5 highest-perplexity questions:")
for result in unigram_question_results[-5:]:
    print(f"{result['perplexity']:.2f} | {result['question']}")

#### Per-Question Bigram Perplexity

In [ ]:
bigram_question_results = []

for question, tokens in zip(ko_validation, ko_validation_lm):
    log_probability = 0.0
    num_tokens = 0

    for i in range(1, len(tokens)):
        previous = tokens[i - 1]
        current = tokens[i]

        log_probability += math.log(
            bigram_probability(previous, current)
        )
        num_tokens += 1

    question_perplexity = math.exp(-log_probability / num_tokens)

    bigram_question_results.append({
        "question": question["question"],
        "perplexity": question_perplexity,
    })

bigram_question_results.sort(key=lambda x: x["perplexity"])

print("5 lowest-perplexity questions:")
for result in bigram_question_results[:5]:
    print(f"{result['perplexity']:.2f} | {result['question']}")

print("\n5 highest-perplexity questions:")
for result in bigram_question_results[-5:]:
    print(f"{result['perplexity']:.2f} | {result['question']}")

## Maks - Trigram model with Laplace smoothing

### Trigram

In [ ]:
trigram_counts = Counter()
previous_two_token_counts = Counter()

for tokens in ko_train_lm:
    for i in range(2, len(tokens)):
        previous_two = (tokens[i - 2], tokens[i - 1])
        current = tokens[i]

        trigram_counts[(previous_two, current)] += 1
        previous_two_token_counts[previous_two] += 1

print("Number of unique trigrams:", len(trigram_counts))
print("\nMost common trigrams:")
print(trigram_counts.most_common(10))

In [ ]:
def trigram_probability(previous_two, current):
    count = trigram_counts.get((previous_two, current), 0)
    previous_count = previous_two_token_counts.get(previous_two, 0)

    return (count + 1) / (previous_count + lm_vocab_size)

In [ ]:
log_probability = 0.0
num_tokens = 0

for tokens in ko_validation_lm:
    for i in range(2, len(tokens)):
        previous_two = (tokens[i - 2], tokens[i - 1])
        current = tokens[i]

        log_probability += math.log(
            trigram_probability(previous_two, current)
        )
        num_tokens += 1

average_negative_log_probability = -log_probability / num_tokens
trigram_perplexity = math.exp(average_negative_log_probability)

print("Validation trigram tokens:", num_tokens)
print("Trigram validation perplexity:", trigram_perplexity)

In [ ]:
trigram_question_results = []

for question, tokens in zip(ko_validation, ko_validation_lm):
    log_probability = 0.0
    num_tokens = 0

    for i in range(2, len(tokens)):
        previous_two = (tokens[i - 2], tokens[i - 1])
        current = tokens[i]

        log_probability += math.log(
            trigram_probability(previous_two, current)
        )
        num_tokens += 1

    question_perplexity = math.exp(-log_probability / num_tokens)

    trigram_question_results.append({
        "question": question["question"],
        "perplexity": question_perplexity,
    })

trigram_question_results.sort(key=lambda x: x["perplexity"])

print("5 lowest-perplexity questions:")
for result in trigram_question_results[:5]:
    print(f"{result['perplexity']:.2f} | {result['question']}")

print("\n5 highest-perplexity questions:")
for result in trigram_question_results[-5:]:
    print(f"{result['perplexity']:.2f} | {result['question']}")

In [ ]:
print("-- Korean Language Model Comparison --")
print(f"Unigram perplexity:  {unigram_perplexity:.2f}")
print(f"Bigram perplexity:   {bigram_perplexity:.2f}")
print(f"Trigram perplexity:  {trigram_perplexity:.2f}")

### Summary and Comparison

In [ ]:
unknown_tokens = 0
total_validation_tokens = 0

for example in ko_validation:
    for token in example["question_tokens"]:
        total_validation_tokens += 1
        if token not in vocab:
            unknown_tokens += 1

unknown_token_rate = unknown_tokens / total_validation_tokens

bigram_difference = bigram_perplexity - unigram_perplexity
bigram_percentage = (bigram_difference / unigram_perplexity) * 100

trigram_difference = trigram_perplexity - unigram_perplexity
trigram_percentage = (trigram_difference / unigram_perplexity) * 100

trigram_bigram_difference = trigram_perplexity - bigram_perplexity
trigram_bigram_percentage = (
    trigram_bigram_difference / bigram_perplexity
) * 100

print("Korean Language Model Summary")

print("\nDataset:")
print("Training questions:", len(ko_train))
print("Validation questions:", len(ko_validation))

print("\nVocabulary:")
print("Training vocabulary size:", vocab_size)
print("LM vocabulary size:", lm_vocab_size)

print("\nTokens:")
print("Training LM tokens:", lm_total_tokens)
print("Validation question tokens:", total_validation_tokens)
print("Unknown validation tokens:", unknown_tokens)
print("Unknown-token rate:", f"{unknown_token_rate * 100:.2f}%")

print("\nModel results:")
print("Unigram perplexity:", f"{unigram_perplexity:.2f}")
print("Bigram perplexity:", f"{bigram_perplexity:.2f}")
print("Trigram perplexity:", f"{trigram_perplexity:.2f}")

print("\nPerplexity comparisons:")
print(
    f"Bigram vs unigram: {bigram_difference:.2f} "
    f"({bigram_percentage:+.2f}%)"
)
print(
    f"Trigram vs unigram: {trigram_difference:.2f} "
    f"({trigram_percentage:+.2f}%)"
)
print(
    f"Trigram vs bigram: {trigram_bigram_difference:.2f} "
    f"({trigram_bigram_percentage:+.2f}%)"
)

# Week 38 - Language Modelling - Korean (BERT tokenization)

## Imports

In [ ]:
from collections import Counter

## Dataset Preparation

In [ ]:
ds = get_dataset()

## Setup and tokenization

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "google-bert/bert-base-multilingual-cased"
)

ko_train = ds["train"].filter(lambda example: example["lang"] == "ko")
ko_validation = ds["validation"].filter(
    lambda example: example["lang"] == "ko"
)


def tokenize_with_bert(example):
    return {"question_tokens": tokenizer.tokenize(example["question"])}


ko_train = ko_train.map(tokenize_with_bert)
ko_validation = ko_validation.map(tokenize_with_bert)

print("Example BERT tokens:", ko_train[0]["question_tokens"])

## Build Vocab

In [ ]:
vocab = {"<UNK>": 0}

for example in ko_train:
    for token in example["question_tokens"]:
        if token not in vocab:
            vocab[token] = len(vocab)

print("Original vocabulary size:", len(vocab))

START_TOKEN = "<START>"
END_TOKEN = "<END>"


def prepare_lm_tokens(tokens):
    processed = [token if token in vocab else "<UNK>" for token in tokens]
    return [START_TOKEN] + processed + [END_TOKEN]


ko_train_lm = [
    prepare_lm_tokens(example["question_tokens"])
    for example in ko_train
]
ko_validation_lm = [
    prepare_lm_tokens(example["question_tokens"])
    for example in ko_validation
]

lm_vocab_size = len(vocab) + 2

## Unigram Model

In [ ]:
lm_token_counts = Counter()

for tokens in ko_train_lm:
    lm_token_counts.update(tokens)

lm_total_tokens = sum(lm_token_counts.values())

unigram_probs = {}

for token in vocab:
    count = lm_token_counts.get(token, 0)
    unigram_probs[token] = (
        count + 1
    ) / (lm_total_tokens + lm_vocab_size)

for token in [START_TOKEN, END_TOKEN]:
    count = lm_token_counts.get(token, 0)
    unigram_probs[token] = (
        count + 1
    ) / (lm_total_tokens + lm_vocab_size)

log_prob = 0.0
num_tokens = 0

for tokens in ko_validation_lm:
    for token in tokens[1:]:
        log_prob += math.log(unigram_probs[token])
        num_tokens += 1

unigram_perplexity = math.exp(-log_prob / num_tokens)
print(f"Unigram validation perplexity: {unigram_perplexity:.2f}")

## Bigram

In [ ]:
bigram_counts = Counter()
previous_token_counts = Counter()

for tokens in ko_train_lm:
    for i in range(1, len(tokens)):
        prev_token, curr_token = tokens[i - 1], tokens[i]
        bigram_counts[(prev_token, curr_token)] += 1
        previous_token_counts[prev_token] += 1


def bigram_probability(prev_token, curr_token):
    count = bigram_counts.get((prev_token, curr_token), 0)
    prev_count = previous_token_counts.get(prev_token, 0)
    return (count + 1) / (prev_count + lm_vocab_size)


log_prob = 0.0
num_tokens = 0

for tokens in ko_validation_lm:
    for i in range(1, len(tokens)):
        log_prob += math.log(
            bigram_probability(tokens[i - 1], tokens[i])
        )
        num_tokens += 1

bigram_perplexity = math.exp(-log_prob / num_tokens)
print(f"Bigram validation perplexity: {bigram_perplexity:.2f}")

## Trigram Model

In [ ]:
trigram_counts = Counter()
bigram_context_counts = Counter()

for tokens in ko_train_lm:
    for i in range(2, len(tokens)):
        context = (tokens[i - 2], tokens[i - 1])
        curr_token = tokens[i]

        trigram_counts[(context, curr_token)] += 1
        bigram_context_counts[context] += 1


def trigram_probability(context, curr_token):
    count = trigram_counts.get((context, curr_token), 0)
    context_count = bigram_context_counts.get(context, 0)
    return (count + 1) / (context_count + lm_vocab_size)


log_prob = 0.0
num_tokens = 0

for tokens in ko_validation_lm:
    for i in range(2, len(tokens)):
        context = (tokens[i - 2], tokens[i - 1])
        log_prob += math.log(
            trigram_probability(context, tokens[i])
        )
        num_tokens += 1

trigram_perplexity = math.exp(-log_prob / num_tokens)
print(f"Trigram validation perplexity: {trigram_perplexity:.2f}")

## Unknown tokens analysis

In [ ]:
total_val_tokens = sum(len(tokens) - 2 for tokens in ko_validation_lm) # -2 for START/END
unknown_val_tokens = sum(tokens.count("<UNK>") for tokens in ko_validation_lm)

print(f"Total validation tokens: {total_val_tokens}")
print(f"Unknown validation tokens: {unknown_val_tokens}")
print(f"Unknown-token rate: {unknown_val_tokens / total_val_tokens * 100:.2f}%")

## Perplexity comparisons

### Unigram

In [ ]:
unigram_question_results = []

for question, tokens in zip(ko_validation, ko_validation_lm):
    log_probability = 0.0
    num_tokens = 0

    for token in tokens[1:]:
        log_probability += math.log(unigram_probs[token])
        num_tokens += 1

    question_perplexity = math.exp(-log_probability / num_tokens)

    unigram_question_results.append({
        "question": question["question"],
        "perplexity": question_perplexity,
    })

unigram_question_results.sort(key=lambda x: x["perplexity"])

print("5 lowest-perplexity questions:")
for result in unigram_question_results[:5]:
    print(f"{result['perplexity']:.2f} | {result['question']}")

print("\n5 highest-perplexity questions:")
for result in unigram_question_results[-5:]:
    print(f"{result['perplexity']:.2f} | {result['question']}")

### Bigram

In [ ]:
bigram_question_results = []

for question, tokens in zip(ko_validation, ko_validation_lm):
    log_probability = 0.0
    num_tokens = 0

    for i in range(1, len(tokens)):
        previous = tokens[i - 1]
        current = tokens[i]

        log_probability += math.log(
            bigram_probability(previous, current)
        )
        num_tokens += 1

    question_perplexity = math.exp(-log_probability / num_tokens)

    bigram_question_results.append({
        "question": question["question"],
        "perplexity": question_perplexity,
    })

bigram_question_results.sort(key=lambda x: x["perplexity"])

print("5 lowest-perplexity questions:")
for result in bigram_question_results[:5]:
    print(f"{result['perplexity']:.2f} | {result['question']}")

print("\n5 highest-perplexity questions:")
for result in bigram_question_results[-5:]:
    print(f"{result['perplexity']:.2f} | {result['question']}")

### Trigram

In [ ]:
trigram_question_results = []

for question, tokens in zip(ko_validation, ko_validation_lm):
    log_probability = 0.0
    num_tokens = 0

    for i in range(2, len(tokens)):
        previous_two = (tokens[i - 2], tokens[i - 1])
        current = tokens[i]

        log_probability += math.log(
            trigram_probability(previous_two, current)
        )
        num_tokens += 1

    question_perplexity = math.exp(-log_probability / num_tokens)

    trigram_question_results.append({
        "question": question["question"],
        "perplexity": question_perplexity,
    })

trigram_question_results.sort(key=lambda x: x["perplexity"])

print("5 lowest-perplexity questions:")
for result in trigram_question_results[:5]:
    print(f"{result['perplexity']:.2f} | {result['question']}")

print("\n5 highest-perplexity questions:")
for result in trigram_question_results[-5:]:
    print(f"{result['perplexity']:.2f} | {result['question']}")

### Comparison

In [ ]:
vocab_size = len(vocab)

unknown_tokens = 0
total_validation_tokens = 0

for example in ko_validation:
    for token in example["question_tokens"]:
        total_validation_tokens += 1
        if token not in vocab:
            unknown_tokens += 1

unknown_token_rate = unknown_tokens / total_validation_tokens

bigram_difference = bigram_perplexity - unigram_perplexity
bigram_percentage = (bigram_difference / unigram_perplexity) * 100

trigram_difference = trigram_perplexity - unigram_perplexity
trigram_percentage = (trigram_difference / unigram_perplexity) * 100

trigram_bigram_difference = trigram_perplexity - bigram_perplexity
trigram_bigram_percentage = (
    trigram_bigram_difference / bigram_perplexity
) * 100

print("Korean Language Model Summary")

print("\nDataset:")
print("Training questions:", len(ko_train))
print("Validation questions:", len(ko_validation))

print("\nVocabulary:")
print("Training vocabulary size:", vocab_size)
print("LM vocabulary size:", lm_vocab_size)

print("\nTokens:")
print("Training LM tokens:", lm_total_tokens)
print("Validation question tokens:", total_validation_tokens)
print("Unknown validation tokens:", unknown_tokens)
print("Unknown-token rate:", f"{unknown_token_rate * 100:.2f}%")

print("\nModel results:")
print("Unigram perplexity:", f"{unigram_perplexity:.2f}")
print("Bigram perplexity:", f"{bigram_perplexity:.2f}")
print("Trigram perplexity:", f"{trigram_perplexity:.2f}")

print("\nPerplexity comparisons:")
print(
    f"Bigram vs unigram: {bigram_difference:.2f} "
    f"({bigram_percentage:+.2f}%)"
)
print(
    f"Trigram vs unigram: {trigram_difference:.2f} "
    f"({trigram_percentage:+.2f}%)"
)
print(
    f"Trigram vs bigram: {trigram_bigram_difference:.2f} "
    f"({trigram_bigram_percentage:+.2f}%)"
)

# W39 - Learned Answerability

## Dataset Preparation

In [ ]:
ds = get_dataset()
ds = nltk_tokenize_dataset(ds)


## Pavlos - Interpretable Learned Baseline (logistic regression)

### Imports

In [ ]:
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import hstack, csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

### Data preparation

In [ ]:
train_data = ds["train"]
validation_data = ds["validation"]

print("Training examples:", len(train_data))
print("Validation examples:", len(validation_data))

print("Training languages:",
      set(train_data["lang"]))
print("Validation languages:",
      set(validation_data["lang"]))

In [ ]:
y_train = [int(example["answerable"]) for example in train_data]
y_validation = [int(example["answerable"]) for example in validation_data]

print("Training labels:", Counter(y_train))
print("Validation labels:", Counter(y_validation))

train_questions = [example["question"] for example in train_data]
train_contexts = [example["context"] for example in train_data]

validation_questions = [example["question"] for example in validation_data]
validation_contexts = [example["context"] for example in validation_data]

print("Training questions:", len(train_questions))
print("Training contexts:", len(train_contexts))
print("Validation questions:", len(validation_questions))
print("Validation contexts:", len(validation_contexts))

print("\nExample question:", train_questions[0])
print("\nExample context:", train_contexts[0][:200])

vectorizer = CountVectorizer()

vectorizer.fit(train_questions + train_contexts)

print("Vocabulary size:", len(vectorizer.vocabulary_))

In [ ]:
Xq_train = vectorizer.transform(train_questions)
Xc_train = vectorizer.transform(train_contexts)

Xq_validation = vectorizer.transform(validation_questions)
Xc_validation = vectorizer.transform(validation_contexts)

print("Question training matrix:", Xq_train.shape)
print("Context training matrix:", Xc_train.shape)
print("Question validation matrix:", Xq_validation.shape)
print("Context validation matrix:", Xc_validation.shape)

In [ ]:
def extract_extra_features(questions, contexts):
    features = []

    for question, context in zip(questions, contexts):
        question_words = set(question.lower().split())
        context_words = set(context.lower().split())

        overlap_count = len(question_words & context_words)
        overlap_ratio = overlap_count / max(len(question_words), 1)

        question_length = len(question.split())
        context_length = len(context.split())
        length_ratio = question_length / max(context_length, 1)

        features.append([
            overlap_count,
            overlap_ratio,
            question_length,
            context_length,
            length_ratio
        ])

    return np.array(features)

extra_train = extract_extra_features(train_questions, train_contexts)
extra_validation = extract_extra_features(
    validation_questions, validation_contexts
)

print("Extra training features:", extra_train.shape)
print("Extra validation features:", extra_validation.shape)
print("Example feature row:", extra_train[0])

In [ ]:
X_train = hstack([
    Xq_train,
    Xc_train,
    csr_matrix(extra_train)
]).tocsr()

X_validation = hstack([
    Xq_validation,
    Xc_validation,
    csr_matrix(extra_validation)
]).tocsr()

print("Final training matrix:", X_train.shape)
print("Final validation matrix:", X_validation.shape)

### Model training

In [ ]:
clf = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
    verbose=1
)

print("Starting logistic regression training...", flush=True)

clf.fit(X_train, y_train)

print("\nTraining completed!")
print("Iterations used:", clf.n_iter_[0])
print("Training accuracy:", clf.score(X_train, y_train))

### Evaluation

In [ ]:
y_pred = clf.predict(X_validation)

print("Accuracy:", accuracy_score(y_validation, y_pred))
print("Precision:", precision_score(y_validation, y_pred, zero_division=0))
print("Recall:", recall_score(y_validation, y_pred, zero_division=0))
print("F1:", f1_score(y_validation, y_pred, zero_division=0))

print("\nConfusion matrix:")
print(confusion_matrix(y_validation, y_pred))

print("\nClassification report:")
print(classification_report(
    y_validation,
    y_pred,
    target_names=["Unanswerable", "Answerable"],
    zero_division=0
))

The BoW logistic model is very good at recognizing answerable questions, but struggles to identify the minority unanswerable class.

## Maks - Recurrent Neural Classifier

### Imports

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
import torch.nn as nn
import time
from sklearn.metrics import classification_report, f1_score

### Vocab and data preparation

In [ ]:
PAD = "<PAD>"
UNK = "<UNK>"


def generate_vocab(ds):
    vocab = {PAD: 0, UNK: 1}

    for example in ds["train"]:
        for token in example["question_tokens"] + example["context_tokens"]:
            if token not in vocab:
                vocab[token] = len(vocab)

    return vocab


vocab = generate_vocab(ds)

vectorized_seqs = [
    [vocab.get(token, vocab[UNK]) for token in example["context_tokens"]]
    for example in ds["train"]
]

embed = torch.nn.Embedding(len(vocab), 4)
lstm = torch.nn.LSTM(
    input_size=4,
    hidden_size=8,
    num_layers=1,
    batch_first=True,
)

max_len = max(len(seq) for seq in vectorized_seqs)
padded_seqs = [
    seq + [vocab[PAD]] * (max_len - len(seq))
    for seq in vectorized_seqs
]

for seq in padded_seqs[:3]:
    print(seq)

sorted_indices = sorted(
    range(len(padded_seqs)),
    key=lambda i: len(padded_seqs[i]),
    reverse=True,
)
sorted_seqs = [padded_seqs[i] for i in sorted_indices]

packed_seqs = torch.nn.utils.rnn.pack_padded_sequence(
    torch.tensor(sorted_seqs, dtype=torch.long),
    lengths=[len(seq) for seq in sorted_seqs],
    batch_first=True,
)

embedded_seqs = embed(packed_seqs.data)

packed_embedded_seqs = torch.nn.utils.rnn.PackedSequence(
    embedded_seqs,
    packed_seqs.batch_sizes,
    packed_seqs.sorted_indices,
    packed_seqs.unsorted_indices,
)

lstm_out, (h_n, c_n) = lstm(packed_embedded_seqs)
unpacked_lstm_out, _ = torch.nn.utils.rnn.pad_packed_sequence(
    lstm_out,
    batch_first=True,
)

In [ ]:
class AnswerabilityDataset(Dataset):
    def __init__(self, split, vocab):
        self.examples = split
        self.vocab = vocab

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        example = self.examples[idx]

        question = [
            self.vocab.get(t, self.vocab[UNK])
            for t in example["question_tokens"]
        ]
        context = [
            self.vocab.get(t, self.vocab[UNK])
            for t in example["context_tokens"]
        ]

        label = float(example["answerable"])

        return (
            torch.tensor(question, dtype=torch.long),
            torch.tensor(context, dtype=torch.long),
            torch.tensor(label, dtype=torch.float32),
        )


def collate_batch(batch):
    questions, contexts, labels = zip(*batch)

    q_lengths = torch.tensor([len(q) for q in questions])
    c_lengths = torch.tensor([len(c) for c in contexts])

    questions = pad_sequence(
        questions, batch_first=True, padding_value=vocab[PAD]
    )
    contexts = pad_sequence(
        contexts, batch_first=True, padding_value=vocab[PAD]
    )

    labels = torch.stack(labels)

    return questions, q_lengths, contexts, c_lengths, labels


train_data = AnswerabilityDataset(ds["train"], vocab)
val_data = AnswerabilityDataset(ds["validation"], vocab)

train_loader = DataLoader(
    train_data, batch_size=1024, shuffle=True,
    collate_fn=collate_batch
)
val_loader = DataLoader(
    val_data, batch_size=1024, shuffle=False,
    collate_fn=collate_batch
)

### (unidirectional) LSTM (for comparison)

In [ ]:
class AnswerabilityLSTM(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=64,
        hidden_size=64,
    ):
        super().__init__()

        self.question_embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=0
        )
        self.context_embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=0
        )

        self.question_lstm = nn.LSTM(
            embedding_dim, hidden_size,
            batch_first=True
        )
        self.context_lstm = nn.LSTM(
            embedding_dim, hidden_size,
            batch_first=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size, 1),
        )

    def encode(self, tokens, lengths, embedding, lstm):
        embedded = embedding(tokens)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, (h_n, _) = lstm(packed)

        return h_n[-1]

    def forward(
        self, questions, q_lengths, contexts, c_lengths
    ):
        q_repr = self.encode(
            questions, q_lengths,
            self.question_embedding, self.question_lstm
        )

        c_repr = self.encode(
            contexts, c_lengths,
            self.context_embedding, self.context_lstm
        )

        combined = torch.cat([q_repr, c_repr], dim=1)

        return self.classifier(combined).squeeze(1)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = AnswerabilityLSTM(len(vocab)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

#### Model training

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for questions, q_lengths, contexts, c_lengths, labels in loader:
        questions = questions.to(device)
        contexts = contexts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(
            questions, q_lengths, contexts, c_lengths
        )

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    return total_loss / total_examples


start_time = time.time()
epoch_times = []
for epoch in range(10):
    current_epoch_time = time.time()
    loss = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    print(f"Epoch {epoch + 1}: loss = {loss:.4f}")
    epoch_times.append(time.time() - current_epoch_time)
    print(f"Last epoch took {epoch_times[-1]:.2f} seconds")
print(f"Average epoch time: {sum(epoch_times) / len(epoch_times):.2f} seconds")

#### Evaluation

In [ ]:
@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    all_labels = []
    all_preds = []
    all_probs = []

    for questions, q_lengths, contexts, c_lengths, labels in loader:
        logits = model(
            questions.to(device), q_lengths,
            contexts.to(device), c_lengths
        )

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long()

        all_labels.extend(labels.long().tolist())
        all_preds.extend(preds.cpu().tolist())
        all_probs.extend(probs.cpu().tolist())

    return all_labels, all_preds, all_probs


y_true, y_pred, y_prob = predict(model, val_loader, device)

report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["Unanswerable", "Answerable"],
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).T
display(report_df.round(3))

print(f"Macro-F1: {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")

In [ ]:
results_df = pd.DataFrame({
    "true_label": y_true,
    "predicted_label": y_pred,
    "answerable_probability": y_prob,
})

display(
    results_df.groupby("true_label")["answerable_probability"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(3)
)

### Bi-directional LSTM (considered main submission)

In [ ]:
class AnswerabilityLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_size=64):
        super().__init__()

        self.question_embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=0
        )
        self.context_embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=0
        )

        self.question_lstm = nn.LSTM(
            embedding_dim,
            hidden_size,
            batch_first=True,
            bidirectional=True,
        )
        self.context_lstm = nn.LSTM(
            embedding_dim,
            hidden_size,
            batch_first=True,
            bidirectional=True,
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 4, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size, 1),
        )

    def encode(self, tokens, lengths, embedding, lstm):
        embedded = embedding(tokens)
        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, (hidden, _) = lstm(packed)
        forward = hidden[-2]
        backward = hidden[-1]

        return torch.cat([forward, backward], dim=1)

    def forward(self, questions, q_lengths, contexts, c_lengths):
        q_repr = self.encode(
            questions, q_lengths, self.question_embedding, self.question_lstm
        )
        c_repr = self.encode(
            contexts, c_lengths, self.context_embedding, self.context_lstm
        )

        combined = torch.cat([q_repr, c_repr], dim=1)
        return self.classifier(combined).squeeze(1)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AnswerabilityLSTM(len(vocab)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

#### Model training

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for questions, q_lengths, contexts, c_lengths, labels in loader:
        questions = questions.to(device)
        contexts = contexts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(
            questions, q_lengths, contexts, c_lengths
        )

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    return total_loss / total_examples


start_time = time.time()
epoch_times = []
for epoch in range(10):
    current_epoch_time = time.time()
    loss = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    print(f"Epoch {epoch + 1}: loss = {loss:.4f}")
    epoch_times.append(time.time() - current_epoch_time)
    print(f"Last epoch took {epoch_times[-1]:.2f} seconds")
print(f"Average epoch time: {sum(epoch_times) / len(epoch_times):.2f} seconds")

#### Evaluation

In [ ]:
@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    all_labels = []
    all_preds = []
    all_probs = []

    for questions, q_lengths, contexts, c_lengths, labels in loader:
        logits = model(
            questions.to(device), q_lengths,
            contexts.to(device), c_lengths
        )

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long()

        all_labels.extend(labels.long().tolist())
        all_preds.extend(preds.cpu().tolist())
        all_probs.extend(probs.cpu().tolist())

    return all_labels, all_preds, all_probs


y_true, y_pred, y_prob = predict(model, val_loader, device)

report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["Unanswerable", "Answerable"],
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).T
display(report_df.round(3))

print(f"Macro-F1: {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")

In [ ]:
results_df = pd.DataFrame({
    "true_label": y_true,
    "predicted_label": y_pred,
    "answerable_probability": y_prob,
})

display(
    results_df.groupby("true_label")["answerable_probability"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(3)
)

In [ ]:
contexts = [example["context"] for example in ds["validation"]]

contexts = sorted(contexts, key=len, reverse=True)

for i, context in enumerate(contexts[:10]):
    print(f"Context {i + 1}: {len(context.split())} words")

avg_context_length = sum(len(context.split()) for context in contexts) / len(contexts)
print(f"Average context length: {avg_context_length:.2f} words")

## Save models for later

In [ ]:
import joblib
import json

# 1. Save Logistic Regression artifacts
# (Assuming your vectorizer is named 'vectorizer' and classifier is 'clf')
joblib.dump(clf, 'week39_lr_model.pkl')
joblib.dump(vectorizer, 'week39_lr_vectorizer.pkl') 

# 2. Save LSTM artifacts
torch.save(model.state_dict(), 'week39_lstm_model.pt')
with open('week39_vocab.json', 'w') as f:
    json.dump(vocab, f)
    
print("Saved Week 39 models and vocabularies")

# W40 - Multilingual QA

## Imports

In [ ]:
!pip install evaluate --quiet
import evaluate
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import os
from sklearn.metrics import precision_recall_fscore_support, f1_score
from datasets import load_dataset, concatenate_datasets, DatasetDict
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
model.to(device)

## Data Preparation

In [ ]:
ds = get_dataset()

### Tokenisation

In [ ]:
MAX_LENGTH = 384
STRIDE = 128

def tokenize_examples(examples):
    return tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
    )

### Answer Positions Alignment

In [ ]:
def align_answer_positions(
    input_ids, offsets, sequence_ids,
    answer, answer_start, answerable, cls_token_id,
):
    cls_index = input_ids.index(cls_token_id)
    if not answerable:
        return cls_index, cls_index

    answer_end = answer_start + len(answer)
    context_positions = [i for i, s in enumerate(sequence_ids) if s == 1]
    if not context_positions:
        return cls_index, cls_index

    first, last = context_positions[0], context_positions[-1]
    if offsets[first][0] > answer_start or offsets[last][1] < answer_end:
        return cls_index, cls_index

    start_token = first
    while start_token <= last and offsets[start_token][1] <= answer_start:
        start_token += 1
    end_token = last
    while end_token >= first and offsets[end_token][0] >= answer_end:
        end_token -= 1

    if start_token > last or end_token < first:
        return cls_index, cls_index
    return start_token, end_token

### Preprocessing

In [ ]:
def preprocess_examples(examples):
    tokenized = tokenize_examples(examples)

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized["offset_mapping"]

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_index = sample_mapping[i]

        # Use updated alignment function
        start, end = align_answer_positions(
            input_ids=tokenized["input_ids"][i],
            offsets=offsets,
            sequence_ids=tokenized.sequence_ids(i),
            answer=examples["answer"][sample_index],
            answer_start=examples["answer_start"][sample_index],
            answerable=examples["answerable"][sample_index],
            cls_token_id=tokenizer.cls_token_id,
        )

        start_positions.append(start)
        end_positions.append(end)

    tokenized.pop("offset_mapping")
    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions

    return tokenized

In [ ]:
train_features = ds["train"].map(
    preprocess_examples,
    batched=True,
    remove_columns=ds["train"].column_names,
)

validation_features = ds["validation"].map(
    preprocess_examples,
    batched=True,
    remove_columns=ds["validation"].column_names,
)

### Validate alignments

In [ ]:
for split in ["train", "validation"]:
    errors = 0
    answerable_count = 0
    unanswerable_count = 0

    for example in ds[split]:
        if not example["answerable"]:
            unanswerable_count += 1
            continue

        answerable_count += 1
        context = example["context"]
        answer = example["answer"]
        start = example["answer_start"]

        if start < 0 or context[start:start + len(answer)] != answer:
            errors += 1

    print(f"\n{split}")
    print(f"Answerable: {answerable_count}")
    print(f"Unanswerable: {unanswerable_count}")
    print(f"Alignment errors: {errors}")
    print(
        f"Misaligned ratio: {errors / answerable_count:.2%}"
        if answerable_count else "Misaligned ratio: N/A"
    )

## Fine-tuning

In [ ]:
SAVE_DIR = "./xlmr-qa-final"

# 1. Check if the model has already been trained and saved
if os.path.exists(SAVE_DIR) and os.path.exists(os.path.join(SAVE_DIR, "config.json")):
    print(f"Found existing trained model at {SAVE_DIR}. Loading instead of training...")
    # Overwrite the untrained model and tokenizer with the saved ones
    model = AutoModelForQuestionAnswering.from_pretrained(SAVE_DIR)
    tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
    model = model.to(device)

else:
    print(f"No trained model found at {SAVE_DIR}. Starting training...")
    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer
    )

    training_args = TrainingArguments(
        output_dir="./xlmr-qa-checkpoints",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        dataloader_pin_memory=torch.cuda.is_available(),
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        seed=42,
    )

    model = model.to(torch.float32)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_features,
        eval_dataset=validation_features,
        data_collator=data_collator,
        processing_class=tokenizer,
    )

    trainer.train()
    trainer.save_model("./xlmr-qa-final")
    tokenizer.save_pretrained("./xlmr-qa-final")

## Evaluation

### SQuAD-v2

In [ ]:
def predict_qa(
    question,
    context,
    model,
    tokenizer,
    max_length=384,
    stride=128,
    max_answer_length=30,
):
    model.eval()

    encoded = tokenizer(
        question,
        context,
        truncation="only_second",
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding=True,
        return_tensors="pt",
    )

    offsets = encoded.pop("offset_mapping")
    encoded.pop("overflow_to_sample_mapping")

    device = next(model.parameters()).device
    model_inputs = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    with torch.no_grad():
        outputs = model(**model_inputs)

    best_span_score = float("-inf")
    best_answer = ""
    best_start = -1
    best_end = -1
    best_null_score = float("-inf")

    for i in range(len(model_inputs["input_ids"])):
        input_ids = model_inputs["input_ids"][i].tolist()
        start_logits = outputs.start_logits[i]
        end_logits = outputs.end_logits[i]

        cls_index = input_ids.index(tokenizer.cls_token_id)
        null_score = (
            start_logits[cls_index].item()
            + end_logits[cls_index].item()
        )
        best_null_score = max(best_null_score, null_score)

        sequence_ids = encoded.sequence_ids(i)
        window_offsets = offsets[i].tolist()

        context_positions = [
            j
            for j, seq_id in enumerate(sequence_ids)
            if seq_id == 1
            and window_offsets[j][1] > window_offsets[j][0]
        ]

        if not context_positions:
            continue

        top_starts = torch.topk(start_logits, min(20, len(start_logits))).indices.tolist()
        top_ends = torch.topk(end_logits, min(20, len(end_logits))).indices.tolist()

        valid_positions = set(context_positions)

        for start_idx in top_starts:
            if start_idx not in valid_positions:
                continue

            for end_idx in top_ends:
                if end_idx not in valid_positions or end_idx < start_idx:
                    continue
                if end_idx - start_idx + 1 > max_answer_length:
                    continue

                char_start = window_offsets[start_idx][0]
                char_end = window_offsets[end_idx][1]

                if char_end <= char_start:
                    continue

                score = (
                    start_logits[start_idx].item()
                    + end_logits[end_idx].item()
                )

                if score > best_span_score:
                    best_span_score = score
                    best_answer = context[char_start:char_end]
                    best_start = char_start
                    best_end = char_end

    if best_span_score == float("-inf"):
        return {
            "best_answer": "",
            "answer_start": -1,
            "answer_end": -1,
            "no_answer_score": float("inf"),
        }

    return {
        "best_answer": best_answer,
        "answer_start": best_start,
        "answer_end": best_end,
        "no_answer_score": best_null_score - best_span_score,
    }

In [ ]:
def add_qa_predictions(
    dataset,
    model,
    tokenizer,
    max_length=384,
    stride=128,
    max_answer_length=30,
):
    def predict_batch(batch):
        predictions = [
            predict_qa(
                question=question,
                context=context,
                model=model,
                tokenizer=tokenizer,
                max_length=max_length,
                stride=stride,
                max_answer_length=max_answer_length,
            )
            for question, context in zip(
                batch["question"], batch["context"]
            )
        ]

        return {
            "predicted_answer": [
                p["best_answer"] for p in predictions
            ],
            "predicted_answer_start": [
                p["answer_start"] for p in predictions
            ],
            "predicted_answer_end": [
                p["answer_end"] for p in predictions
            ],
            "no_answer_score": [
                p["no_answer_score"] for p in predictions
            ],
        }

    return dataset.map(
        predict_batch,
        batched=True,
        batch_size=8,
    )

In [ ]:
predicted_ds = add_qa_predictions(
    ds,
    model,
    tokenizer,
    max_length=MAX_LENGTH,
    stride=STRIDE,
)

In [ ]:
for i in range(5):
    print("Question:", predicted_ds["validation"][i]["question"])
    print("Gold:", predicted_ds["validation"][i]["answer"])
    print("Predicted:", predicted_ds["validation"][i]["predicted_answer"])
    print("No-answer score:", predicted_ds["validation"][i]["no_answer_score"])
    print()

print("Checking unanswerable")
unanswerable = predicted_ds["validation"].filter(lambda example: not example["answerable"])
for i in range(min(5, len(unanswerable))):
    print("Question:", unanswerable[i]["question"])
    print("Gold:", unanswerable[i]["answer"])
    print("Predicted:", unanswerable[i]["predicted_answer"])
    print("No-answer score:", unanswerable[i]["no_answer_score"])
    print()


In [ ]:
squad_metric = evaluate.load("squad_v2")


def make_squad_inputs(dataset, threshold):
    predictions = []
    references = []

    for i, example in enumerate(dataset):
        example_id = str(i)
        answer = example["predicted_answer"]

        if example["no_answer_score"] > threshold:
            answer = ""

        predictions.append({
            "id": example_id,
            "prediction_text": answer,
            "no_answer_probability": 0.0,
        })

        is_answerable = example["answerable"]

        references.append({
            "id": example_id,
            "answers": {
                "text": [example["answer"]] if is_answerable else [],
                "answer_start": (
                    [example["answer_start"]] if is_answerable else []
                ),
            },
        })

    return predictions, references

In [ ]:
def tune_no_answer_threshold(validation_dataset, n_thresholds=101):
    scores = np.asarray(
        validation_dataset["no_answer_score"],
        dtype=float,
    )
    finite_scores = scores[np.isfinite(scores)]

    if len(finite_scores) == 0:
        raise ValueError("No finite no-answer scores found.")

    thresholds = np.linspace(
        finite_scores.min() - 1e-6,
        finite_scores.max() + 1e-6,
        n_thresholds,
    )

    best_threshold = None
    best_f1 = -1
    best_results = None

    for threshold in thresholds:
        predictions, references = make_squad_inputs(
            validation_dataset,
            threshold,
        )
        results = squad_metric.compute(
            predictions=predictions,
            references=references,
        )

        if results["f1"] > best_f1:
            best_f1 = results["f1"]
            best_threshold = float(threshold)
            best_results = results

    return best_threshold, best_results

In [ ]:
best_threshold, validation_results = tune_no_answer_threshold(
    predicted_ds["validation"]
)

print("Selected threshold:", best_threshold)
print("Validation SQuAD-v2:", validation_results)

In [ ]:
def evaluate_squad_split(dataset, threshold):
    predictions, references = make_squad_inputs(
        dataset,
        threshold,
    )

    return squad_metric.compute(
        predictions=predictions,
        references=references,
    )


test_results = evaluate_squad_split(
    predicted_ds["validation"],
    best_threshold,
)

print("Test SQuAD-v2:", test_results)

In [ ]:
def evaluate_answerability(dataset, threshold):
    true_labels = []
    pred_labels = []
    
    for example in dataset:
        true_labels.append(1 if example["answerable"] else 0)

        is_pred_answerable = 1 if example["no_answer_score"] <= threshold else 0
        pred_labels.append(is_pred_answerable)

    precision, recall, f1, support = precision_recall_fscore_support(
        true_labels, pred_labels, labels=[0, 1], zero_division=0
    )
    macro_f1 = f1_score(true_labels, pred_labels, average='macro', zero_division=0)

    return {
        "macro_f1": macro_f1,
        "unanswerable_precision": precision[0],
        "unanswerable_recall": recall[0],
        "unanswerable_f1": f1[0],
        "answerable_precision": precision[1],
        "answerable_recall": recall[1],
        "answerable_f1": f1[1],
    }

def run_full_evaluation(dataset, threshold):
    languages = ["ar", "ko", "te"]
    splits = {"Overall": dataset}
    
    for lang in languages:
        splits[lang] = dataset.filter(lambda x: x["lang"] == lang)

    for name, split_ds in splits.items():
        print(f"Evaluation for: {name.upper()} (N={len(split_ds)})")
        
        if len(split_ds) == 0:
            print("No examples found for this split.")
            continue

        squad_res = evaluate_squad_split(split_ds, threshold)
        print(f"SQuAD-v2 Metrics:")
        print(f"Exact Match {squad_res['exact']:.2f}")
        print(f"Token F1: {squad_res['f1']:.2f}")

        ans_res = evaluate_answerability(split_ds, threshold)
        print(f"Answerability Metrics:")
        print(f"Macro-F1: {ans_res['macro_f1']:.2f}")
        print(f"Answerable- P: {ans_res['answerable_precision']:.2f} | R: {ans_res['answerable_recall']:.2f} | F1: {ans_res['answerable_f1']:.2f}")
        print(f"Unanswerable- P: {ans_res['unanswerable_precision']:.2f} | R: {ans_res['unanswerable_recall']:.2f} | F1: {ans_res['unanswerable_f1']:.2f}")

run_full_evaluation(predicted_ds["validation"], best_threshold)

overall_squad = evaluate_squad_split(predicted_ds["validation"], best_threshold)
overall_ans = evaluate_answerability(predicted_ds["validation"], best_threshold)
combined_results = {**overall_squad, **overall_ans}

results_df = pd.DataFrame([combined_results]).transpose()
display(results_df)
latex_table = results_df.to_latex(index=False, header=True, float_format="%.2f")
print(latex_table)

### Comparison with Sequence labeller

In [ ]:
from sklearn.metrics import cohen_kappa_score
import pandas as pd

qa_model_name = "xlm-roberta-base"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
bio_tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-multilingual-cased")

bio_model = QuestionConditioned().to(device)
bio_model.load_state_dict(torch.load("span_prediction_42.pt"))
bio_model.eval()

def get_qa_span_for_example(ex, qa_model, qa_tokenizer, threshold):
    qa_preds = predict_qa(
        question=ex["question"],
        context=ex["context"],
        model=qa_model,
        tokenizer=qa_tokenizer,
    )
    if qa_preds["no_answer_score"] > threshold:
        return "" 
    return qa_preds["best_answer"]

def get_bio_span_for_example(ex, bio_model, bio_tokenizer):
    return predict_span(
        model=bio_model, 
        tokenizer=bio_tokenizer, 
        question=ex["question"], 
        context=ex["context"], 
        device=device
    )

def calculate_token_f1(pred, gold):
    pred_tokens = pred.split()
    gold_tokens = gold.split()
    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0
    
    common = set(pred_tokens) & set(gold_tokens)
    num_same = len(common)
    if num_same == 0:
        return 0.0
    
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * (precision * recall) / (precision + recall)

print("\n" + "="*60)
print("COMPARING WEEK 40/41 SPAN PREDICTIONS VS WEEK 37 BIO TAGGER")
print("="*60)

comparison_rows = []

for ex in tqdm(ds["validation"], desc="Comparing span predictions"):
    qa_span = get_qa_span_for_example(ex, model, qa_tokenizer, best_threshold)
    bio_span = get_bio_span_for_example(ex, bio_model, bio_tokenizer)
    
    exact_match_agreement = 1 if qa_span == bio_span else 0
    span_token_f1 = calculate_token_f1(qa_span, bio_span)
    
    comparison_rows.append({
        "lang": ex["lang"],
        "exact_match_agreement": exact_match_agreement,
        "span_token_f1": span_token_f1,
        "qa_span": qa_span,
        "bio_span": bio_span,
        "gold_answer": ex["answer"] if ex["answerable"] else ""
    })

df_compare = pd.DataFrame(comparison_rows)

def aggregate_comparison(df):
    results = []
    for lang in ["Overall", "ar", "ko", "te"]:
        subset = df if lang == "Overall" else df[df["lang"] == lang]
        if len(subset) == 0: continue
        results.append({
            "Language": lang,
            "N": len(subset),
            "Span EM Agreement": subset["exact_match_agreement"].mean(),
            "Span Token F1": subset["span_token_f1"].mean()
        })
    return pd.DataFrame(results)

df_results = aggregate_comparison(df_compare)
print("\nSpan Prediction Comparison Summary:")
display(df_results)

print("\nLaTeX Table:")
print(df_results.to_latex(index=False, float_format="%.3f"))

#### Interpretation (maybe not needed)

In [ ]:
pure_span_disagreements = df_compare[
    (df_compare["qa_span"] != "") & 
    (df_compare["bio_span"] != "") & 
    (df_compare["exact_match_agreement"] == 0)
]

print(f"\nTotal pure span disagreements to analyze: {len(pure_span_disagreements)}")

for lang in ["ar", "ko", "te"]:
    print(f"\nExamples of Span Disagreements in {lang.upper()}")
    lang_disagreements = pure_span_disagreements[pure_span_disagreements["lang"] == lang]
    
    for i, row in lang_disagreements.head(3).iterrows():
        print(f"Gold: {row['gold_answer']}")
        print(f"QA Model (W40/41):   {row['qa_span']}")
        print(f"BiLSTM Tagger (W37): {row['bio_span']}\n")

### Comparison with Answerability Classifier

In [ ]:
import joblib
import json
import nltk
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

lr_clf = joblib.load('week39_lr_model.pkl')
lr_vectorizer = joblib.load('week39_lr_vectorizer.pkl')

with open('week39_vocab.json', 'r') as f:
    week39_vocab = json.load(f)
    
lstm_model = AnswerabilityLSTM(len(week39_vocab)).to(device)
lstm_model.load_state_dict(torch.load('week39_lstm_model.pt'))
lstm_model.eval()


def get_lr_abstention(ex, clf, vectorizer):
    q_vec = vectorizer.transform([ex["question"]])
    c_vec = vectorizer.transform([ex["context"]])
    
    extra = extract_extra_features([ex["question"]], [ex["context"]])
    
    X = hstack([q_vec, c_vec, csr_matrix(extra)]).tocsr()
    pred = clf.predict(X)[0]
    return pred

def get_lstm_abstention(ex, model, vocab, device):
    q_tokens = [vocab.get(t, vocab["<UNK>"]) for t in ex["question_tokens"]]
    c_tokens = [vocab.get(t, vocab["<UNK>"]) for t in ex["context_tokens"]]
    
    q_tensor = torch.tensor([q_tokens], dtype=torch.long).to(device)
    c_tensor = torch.tensor([c_tokens], dtype=torch.long).to(device)
    q_len = torch.tensor([len(q_tokens)])
    c_len = torch.tensor([len(c_tokens)])
    
    with torch.no_grad():
        logits = model(q_tensor, q_len, c_tensor, c_len)
        prob = torch.sigmoid(logits).item()
        
    return 1 if prob >= 0.5 else 0

print("\n" + "="*60)
print("COMPARING WEEK 40 ABSTENTION VS WEEK 39 CLASSIFIERS")
print("="*60)

comparison_rows = []

val_dataset = nltk_tokenize_dataset(ds)["validation"]

for ex in tqdm(val_dataset, desc="Comparing abstentions"):
    qa_preds = predict_qa(
        question=ex["question"],
        context=ex["context"],
        model=model,
        tokenizer=tokenizer
    )
    qa_abstained = 0 if qa_preds["no_answer_score"] > best_threshold else 1
    
    lr_abstained = get_lr_abstention(ex, lr_clf, lr_vectorizer)
    
    lstm_abstained = get_lstm_abstention(ex, lstm_model, week39_vocab, device)
    
    comparison_rows.append({
        "lang": ex["lang"],
        "true_label": 1 if ex["answerable"] else 0,
        "qa_abstained": qa_abstained,
        "lr_abstained": lr_abstained,
        "lstm_abstained": lstm_abstained
    })

df_abstain = pd.DataFrame(comparison_rows)

from sklearn.metrics import cohen_kappa_score

def calculate_agreement(df, col1, col2):
    agree_pct = (df[col1] == df[col2]).mean()
    kappa = cohen_kappa_score(df[col1], df[col2])
    return agree_pct, kappa

print("\nAgreement Summary (Overall):")
qa_lr_agree, qa_lr_kappa = calculate_agreement(df_abstain, "qa_abstained", "lr_abstained")
qa_lstm_agree, qa_lstm_kappa = calculate_agreement(df_abstain, "qa_abstained", "lstm_abstained")
lr_lstm_agree, lr_lstm_kappa = calculate_agreement(df_abstain, "lr_abstained", "lstm_abstained")

print(f"Week 40 QA vs Week 39 LR:   Agreement={qa_lr_agree:.2%}, Kappa={qa_lr_kappa:.3f}")
print(f"Week 40 QA vs Week 39 LSTM: Agreement={qa_lstm_agree:.2%}, Kappa={qa_lstm_kappa:.3f}")
print(f"Week 39 LR vs Week 39 LSTM: Agreement={lr_lstm_agree:.2%}, Kappa={lr_lstm_kappa:.3f}")

def get_lang_agreement(df, lang):
    subset = df[df["lang"] == lang]
    if len(subset) == 0: return None
    agree, kappa = calculate_agreement(subset, "qa_abstained", "lstm_abstained")
    return len(subset), agree, kappa

print("\nBreakdown by Language (Week 40 QA vs LSTM):")
for lang in ["ar", "ko", "te"]:
    res = get_lang_agreement(df_abstain, lang)
    if res:
        n, agree, kappa = res
        print(f"  {lang.upper()} (N={n}): Agreement={agree:.2%}, Kappa={kappa:.3f}")

false_abstentions = df_abstain[
    (df_abstain["qa_abstained"] == 0) & 
    (df_abstain["lstm_abstained"] == 1)
]
print(f"\nCases where QA abstained but LSTM tried to answer: {len(false_abstentions)}")

# W43 - Extensions

#### Pavlos - No-answer Threshold

##### Hypothesis: Selecting the no-answer threshold to prioritise recall on unanswerable questions will increase unanswerable-question recall compared with selecting the threshold to maximise answerability macro-F1, but may reduce overall answerability macro-F1.

##### Imports

In [ ]:
import torch
import numpy as np
from datasets import load_dataset, concatenate_datasets, DatasetDict
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from sklearn.metrics import f1_score
from collections import Counter
import re
import string

In [ ]:
model_path = "./xlmr-qa-final"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForQuestionAnswering.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("Device:", device)
print("Model loaded successfully")

##### Split Dataset

In [ ]:
from datasets import load_dataset, concatenate_datasets, DatasetDict

def get_dataset():
    ds = load_dataset("coastalcph/tydi_xor_rc")
    return ds.filter(lambda x: x["lang"] in ["ar", "ko", "te"])


def redistribute_dataset(ds):
    all_data = concatenate_datasets(
        [ds["train"], ds["validation"], ds["test"]]
    )

    splits = {}

    for lang in ["ar", "ko", "te"]:
        data = all_data.filter(lambda x: x["lang"] == lang)

        split = data.train_test_split(test_size=0.2, seed=42)
        train_val = split["train"].train_test_split(
            test_size=0.125, seed=42
        )

        splits[lang] = {
            "train": train_val["train"],
            "validation": train_val["test"],
            "test": split["test"]
        }

    result = {}

    for name in ["train", "validation", "test"]:
        result[name] = concatenate_datasets(
            [splits[lang][name] for lang in ["ar", "ko", "te"]]
        ).shuffle(seed=42)

    return DatasetDict(result)


ds = redistribute_dataset(get_dataset())

for name in ["train", "validation", "test"]:
    print(f"\n{name}: {len(ds[name])} examples")
    print(ds[name].to_pandas()["lang"].value_counts().sort_index())

In [ ]:
WEEK43_MAX_LENGTH = 384
WEEK43_STRIDE = 128

def tokenize_examples(examples):
    return tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=WEEK43_MAX_LENGTH,
        stride=WEEK43_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
    )

##### Represent Unanswerable Questions

In [ ]:
def prepare_inference_features(dataset):
    features = []

    for idx, example in enumerate(dataset):
        inputs = tokenizer(
            example["question"],
            example["context"],
            truncation="only_second",
            max_length=WEEK43_MAX_LENGTH,
            stride=WEEK43_STRIDE,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
        )

        for i, input_ids in enumerate(inputs["input_ids"]):
            sequence_ids = inputs.sequence_ids(i)
            offsets = [
                offset if sequence_ids[j] == 1 else None
                for j, offset in enumerate(inputs["offset_mapping"][i])
            ]

            features.append({
                "example_idx": idx,
                "input_ids": input_ids,
                "attention_mask": inputs["attention_mask"][i],
                "offset_mapping": offsets,
            })

    return features


validation_features_inference = prepare_inference_features(ds["validation"])

print("Validation questions:", len(ds["validation"]))
print("Validation windows:", len(validation_features_inference))

##### Generate start/end Logits

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

model_inputs = [
    {"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]}
    for f in validation_features_inference
]

loader = DataLoader(
    model_inputs,
    batch_size=8,
    collate_fn=DataCollatorWithPadding(tokenizer=tokenizer)
)

all_start_logits = []
all_end_logits = []

model.eval()
with torch.no_grad():
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        all_start_logits.extend(outputs.start_logits.cpu().numpy())
        all_end_logits.extend(outputs.end_logits.cpu().numpy())

print("Windows scored:", len(all_start_logits))

In [ ]:
WEEK43_MAX_ANSWER_LENGTH = 30
WEEK43_N_BEST = 20

def get_best_span(start_logits, end_logits, offsets):
    best_score = float("-inf")
    best_span = ""

    starts = np.argsort(start_logits)[-WEEK43_N_BEST:]
    ends = np.argsort(end_logits)[-WEEK43_N_BEST:]

    for start in starts:
        for end in ends:
            if start > end or end - start + 1 > WEEK43_MAX_ANSWER_LENGTH:
                continue
            if offsets[start] is None or offsets[end] is None:
                continue

            start_char = offsets[start][0]
            end_char = offsets[end][1]

            if start_char >= end_char:
                continue

            score = start_logits[start] + end_logits[end]
            if score > best_score:
                best_score = score
                best_span = (start_char, end_char)

    return best_score, best_span


def get_question_scores(dataset, features, start_logits, end_logits):
    scores = []

    for idx, example in enumerate(dataset):
        context = example["context"]
        best_score = float("-inf")
        best_answer = ""
        null_score = float("inf")

        for i, feature in enumerate(features):
            if feature["example_idx"] != idx:
                continue

            start = start_logits[i]
            end = end_logits[i]
            cls = feature["input_ids"].index(tokenizer.cls_token_id)

            null_score = min(null_score, float(start[cls] + end[cls]))

            span_score, span = get_best_span(
                start, end, feature["offset_mapping"]
            )

            if span and span_score > best_score:
                best_score = span_score
                best_answer = context[span[0]:span[1]]

        scores.append({
            "example_idx": idx,
            "lang": example["lang"],
            "gold_answerable": bool(example["answerable"]),
            "gold_answer": example["answer"],
            "predicted_answer": best_answer,
            "no_answer_score": null_score - best_score,
        })

    return scores


validation_scores = get_question_scores(
    ds["validation"],
    validation_features_inference,
    all_start_logits,
    all_end_logits,
)

print("Question scores:", len(validation_scores))
for item in validation_scores[:5]:
    print(item)

In [ ]:
def normalize_answer(text):
    text = text.lower()
    text = "".join(c for c in text if c not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def exact_match(pred, gold):
    return int(normalize_answer(pred) == normalize_answer(gold))


def token_f1(pred, gold):
    pred = normalize_answer(pred).split()
    gold = normalize_answer(gold).split()

    if not pred or not gold:
        return float(pred == gold)

    common = Counter(pred) & Counter(gold)
    same = sum(common.values())

    if same == 0:
        return 0.0

    precision = same / len(pred)
    recall = same / len(gold)
    return 2 * precision * recall / (precision + recall)


def evaluate_threshold(scores, threshold):
    y_true = np.array([s["gold_answerable"] for s in scores])
    y_pred = np.array([s["no_answer_score"] <= threshold for s in scores])

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    unanswerable = ~y_true
    unanswerable_recall = (
        np.mean(~y_pred[unanswerable]) if unanswerable.any() else 0.0
    )

    em, f1 = [], []
    for s, answerable in zip(scores, y_pred):
        pred = s["predicted_answer"] if answerable else ""
        gold = s["gold_answer"] if s["gold_answerable"] else ""
        em.append(exact_match(pred, gold))
        f1.append(token_f1(pred, gold))

    return {
        "threshold": float(threshold),
        "answerability_macro_f1": float(macro_f1),
        "unanswerable_recall": float(unanswerable_recall),
        "EM": float(np.mean(em)),
        "token_F1": float(np.mean(f1)),
    }


def evaluate_threshold_per_lang(scores, per_lang_threshold):
    y_true = np.array([s["gold_answerable"] for s in scores])
    y_pred = np.array([
        s["no_answer_score"] <= per_lang_threshold[s["lang"]]
        for s in scores
    ])

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    unanswerable = ~y_true
    unanswerable_recall = (
        np.mean(~y_pred[unanswerable]) if unanswerable.any() else 0.0
    )

    em, f1 = [], []
    for s, answerable in zip(scores, y_pred):
        pred = s["predicted_answer"] if answerable else ""
        gold = s["gold_answer"] if s["gold_answerable"] else ""
        em.append(exact_match(pred, gold))
        f1.append(token_f1(pred, gold))

    return {
        "answerability_macro_f1": float(macro_f1),
        "unanswerable_recall": float(unanswerable_recall),
        "EM": float(np.mean(em)),
        "token_F1": float(np.mean(f1)),
    }


thresholds = np.concatenate((
    [-np.inf],
    np.unique([s["no_answer_score"] for s in validation_scores]),
    [np.inf],
))

results = [evaluate_threshold(validation_scores, t) for t in thresholds]

baseline_result = max(results, key=lambda r: r["answerability_macro_f1"])

LANG_RECALL_TARGET = 0.85

per_lang_threshold = {}
for lang in ["ar", "ko", "te"]:
    lang_scores = [s for s in validation_scores if s["lang"] == lang]

    lang_thresholds = np.concatenate((
        [-np.inf],
        np.unique([s["no_answer_score"] for s in lang_scores]),
        [np.inf],
    ))
    lang_results = [evaluate_threshold(lang_scores, t) for t in lang_thresholds]

    max_recall = max(r["unanswerable_recall"] for r in lang_results)
    effective_target = min(LANG_RECALL_TARGET, max_recall - 1e-3)

    lang_eligible = [r for r in lang_results if r["unanswerable_recall"] >= effective_target]
    if lang_eligible:
        chosen = max(
            lang_eligible,
            key=lambda r: (
                r["answerability_macro_f1"],
                -abs(r["unanswerable_recall"] - effective_target),
            ),
        )
    else:
        chosen = max(lang_results, key=lambda r: r["answerability_macro_f1"])

    per_lang_threshold[lang] = chosen["threshold"]

print("Baseline (global) threshold:", baseline_result["threshold"])
print("Per-language thresholds:", per_lang_threshold)

##### Validation

In [ ]:
rows = []

for lang in ["ar", "ko", "te"]:
    scores = [s for s in validation_scores if s["lang"] == lang]

    for name, metrics in [
        ("Baseline", evaluate_threshold(scores, baseline_result["threshold"])),
        ("Week 43 extension", evaluate_threshold(scores, per_lang_threshold[lang])),
    ]:
        rows.append({
            "language": lang,
            "strategy": name,
            "n": len(scores),
            "answerability_macro_f1": metrics["answerability_macro_f1"],
            "unanswerable_recall": metrics["unanswerable_recall"],
            "EM": metrics["EM"],
            "token_F1": metrics["token_F1"],
        })

validation_language_df = pd.DataFrame(rows)
display(validation_language_df.round(4))

In [ ]:
test_features_inference = prepare_inference_features(ds["test"])

print("Test questions:", len(ds["test"]))
print("Test windows:", len(test_features_inference))

In [ ]:
test_inputs = [
    {"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]}
    for f in test_features_inference
]

test_loader = DataLoader(
    test_inputs,
    batch_size=8,
    collate_fn=DataCollatorWithPadding(tokenizer=tokenizer)
)

test_start_logits = []
test_end_logits = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)

        test_start_logits.extend(outputs.start_logits.cpu().numpy())
        test_end_logits.extend(outputs.end_logits.cpu().numpy())

print("Test windows scored:", len(test_start_logits))

In [ ]:
test_scores = get_question_scores(
    ds["test"],
    test_features_inference,
    test_start_logits,
    test_end_logits,
)

print("Test question scores:", len(test_scores))
print("Example:", test_scores[0]) # debug

##### Test Results

In [ ]:
baseline_test = evaluate_threshold(test_scores, baseline_result["threshold"])
extension_test = evaluate_threshold_per_lang(test_scores, per_lang_threshold)

test_comparison_df = pd.DataFrame([
    {"Strategy": "Baseline", **baseline_test},
    {"Strategy": "Week 43 extension", **extension_test},
])

display(test_comparison_df.round(4))

##### Per-Language Tests

In [ ]:
rows = []

for lang in ["ar", "ko", "te"]:
    scores = [s for s in test_scores if s["lang"] == lang]

    for name, metrics in [
        ("Baseline", evaluate_threshold(scores, baseline_result["threshold"])),
        ("Week 43 extension", evaluate_threshold(scores, per_lang_threshold[lang])),
    ]:
        rows.append({
            "language": lang,
            "strategy": name,
            "n": len(scores),
            "answerability_macro_f1": metrics["answerability_macro_f1"],
            "unanswerable_recall": metrics["unanswerable_recall"],
            "EM": metrics["EM"],
            "token_F1": metrics["token_F1"],
        })

test_language_df = pd.DataFrame(rows)
display(test_language_df.round(4))

##### Hypothesis Conclusion: The hypothesis is supported by the test results, though the extension did not reach the 90% unanswerable-recall target on the test set.

In [ ]:
latex_df = test_language_df[
    ["language", "strategy", "answerability_macro_f1", "unanswerable_recall"]
].copy()

latex_df.columns = ["Language", "Strategy", "Macro-F1", "Unans. Recall"]

for col in ["Macro-F1", "Unans. Recall"]:
    latex_df[col] = latex_df[col].map(lambda x: f"{x * 100:.2f}\\%")

print(latex_df.to_latex(index=False, escape=False))

## Maks - Translate questions to English before English QA model

### Imports and config

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_recall_fscore_support
import evaluate
from datasets import DatasetDict, load_from_disk, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Paths
W40_DIR = "./xlmr-qa-final"
EXT_DIR = "./roberta-qa-translated-final"
TRANSLATED_DS_DIR = "./translated_ds"

MAX_LENGTH = 384
STRIDE = 128
MAX_ANSWER_LEN = 30
SEED = 42

### Dataset Translation

In [ ]:
def build_translated_dataset(ds, lang_codes):
    for split in ds:
        all_langs = ds[split]["lang"]
        lookup = {}
        for lang in lang_codes:
            idxs = [i for i, l in enumerate(all_langs) if l == lang]
            questions = [ds[split][i]["question"] for i in idxs]
            translations = translate_batch_to_english(questions, lang)
            assert len(translations) == len(idxs), (
                f"{split}/{lang}: expected {len(idxs)}, got {len(translations)}"
            )
            lookup.update(dict(zip(idxs, translations)))
        ds[split] = ds[split].map(
            lambda ex, idx: {"question_translated": lookup[idx]},
            with_indices=True,
        )
    return ds


def swap_to_translated(ds):
    out = DatasetDict()
    for split in ds:
        out[split] = ds[split].map(
            lambda ex: {
                "question_original": ex["question"],
                "question": ex["question_translated"],
            }
        )
    return out



def build_translated(original_ds, overwrite=False):
    if os.path.exists(TRANSLATED_DS_DIR) and not overwrite:
        print(f"Cached translations from: {TRANSLATED_DS_DIR}")
        return load_from_disk(TRANSLATED_DS_DIR)
    print("Translating dataset")
    ds = build_translated_dataset(original_ds, LANG_CODES)
    ds.save_to_disk(TRANSLATED_DS_DIR)
    return ds

### QA Utilities

In [ ]:
def align_answer_positions(
    input_ids, offsets, sequence_ids,
    answer, answer_start, answerable, cls_token_id,
):
    cls_index = input_ids.index(cls_token_id)
    if not answerable:
        return cls_index, cls_index

    answer_end = answer_start + len(answer)
    context_positions = [i for i, s in enumerate(sequence_ids) if s == 1]
    if not context_positions:
        return cls_index, cls_index

    first, last = context_positions[0], context_positions[-1]
    if offsets[first][0] > answer_start or offsets[last][1] < answer_end:
        return cls_index, cls_index

    start_token = first
    while start_token <= last and offsets[start_token][1] <= answer_start:
        start_token += 1
    end_token = last
    while end_token >= first and offsets[end_token][0] >= answer_end:
        end_token -= 1

    if start_token > last or end_token < first:
        return cls_index, cls_index
    return start_token, end_token

def build_qa_features(dataset, tokenizer, max_length=MAX_LENGTH, stride=STRIDE):
    cls_id = tokenizer.cls_token_id

    def preprocess(examples):
        tok = tokenizer(
            examples["question"], examples["context"],
            truncation="only_second",
            max_length=max_length,
            stride=stride,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
        )
        sample_mapping = tok.pop("overflow_to_sample_mapping")
        offset_mapping = tok["offset_mapping"]

        starts, ends = [], []
        for i, offsets in enumerate(offset_mapping):
            si = sample_mapping[i]
            s, e = align_answer_positions(
                input_ids=tok["input_ids"][i],
                offsets=offsets,
                sequence_ids=tok.sequence_ids(i),
                answer=examples["answer"][si],
                answer_start=examples["answer_start"][si],
                answerable=examples["answerable"][si],
                cls_token_id=cls_id,
            )
            starts.append(s)
            ends.append(e)

        tok.pop("offset_mapping")
        tok["start_positions"] = starts
        tok["end_positions"] = ends
        return tok

    return dataset.map(
        preprocess, batched=True, remove_columns=dataset.column_names
    )

In [ ]:
def fine_tune_qa_model(
    train_ds, val_ds, model_name, output_dir,
    num_epochs=3, batch_size=8, lr=2e-5, seed=SEED,
    max_length=MAX_LENGTH, stride=STRIDE,
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForQuestionAnswering.from_pretrained(model_name)

    train_feats = build_qa_features(train_ds, tokenizer, max_length, stride)
    val_feats = build_qa_features(val_ds,   tokenizer, max_length, stride)

    n_non_cls = sum(
        1 for s, e in zip(train_feats["start_positions"], train_feats["end_positions"])
        if not (s == tokenizer.cls_token_id and e == tokenizer.cls_token_id)
    )
    print(f"[{model_name}] non-CLS training features: {n_non_cls} / {len(train_feats)}")
    if n_non_cls == 0:
        raise RuntimeError(
            "All training spans collapsed to CLS — check align_answer_positions."
        )

    args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        dataloader_pin_memory=torch.cuda.is_available(),
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        seed=seed,
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_feats, eval_dataset=val_feats,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        processing_class=tokenizer,
    )
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return tokenizer, trainer.model

### Prediction

In [ ]:
def predict_qa(
    question, context, model, tokenizer,
    max_length=MAX_LENGTH, stride=STRIDE, max_answer_length=MAX_ANSWER_LEN,
):
    model.eval()
    encoded = tokenizer(
        question, context,
        truncation="only_second", max_length=max_length, stride=stride,
        return_overflowing_tokens=True, return_offsets_mapping=True,
        padding=True, return_tensors="pt",
    )
    offsets = encoded.pop("offset_mapping")
    encoded.pop("overflow_to_sample_mapping")

    dev = next(model.parameters()).device
    model_inputs = {k: v.to(dev) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**model_inputs)

    best_span_score = float("-inf")
    best_answer, best_start, best_end = "", -1, -1
    best_null_score = float("-inf")

    for i in range(len(model_inputs["input_ids"])):
        input_ids = model_inputs["input_ids"][i].tolist()
        start_logits = outputs.start_logits[i]
        end_logits = outputs.end_logits[i]

        cls_index = input_ids.index(tokenizer.cls_token_id)
        null_score = start_logits[cls_index].item() + end_logits[cls_index].item()
        best_null_score = max(best_null_score, null_score)

        sequence_ids = encoded.sequence_ids(i)
        window_offsets = offsets[i].tolist()
        context_positions = [
            j for j, s in enumerate(sequence_ids)
            if s == 1 and window_offsets[j][1] > window_offsets[j][0]
        ]
        if not context_positions:
            continue

        top_starts = torch.topk(start_logits, min(20, len(start_logits))).indices.tolist()
        top_ends = torch.topk(end_logits, min(20, len(end_logits))).indices.tolist()
        valid = set(context_positions)

        for s_idx in top_starts:
            if s_idx not in valid:
                continue
            for e_idx in top_ends:
                if e_idx not in valid or e_idx < s_idx:
                    continue
                if e_idx - s_idx + 1 > max_answer_length:
                    continue
                cs = window_offsets[s_idx][0]
                ce = window_offsets[e_idx][1]
                if ce <= cs:
                    continue
                score = start_logits[s_idx].item() + end_logits[e_idx].item()
                if score > best_span_score:
                    best_span_score = score
                    best_answer = context[cs:ce]
                    best_start = cs
                    best_end = ce

    if best_span_score == float("-inf"):
        return {"best_answer": "", "answer_start": -1, "answer_end": -1,
                "no_answer_score": float("inf")}

    return {
        "best_answer": best_answer,
        "answer_start": best_start,
        "answer_end": best_end,
        "no_answer_score": best_null_score - best_span_score,
    }


def add_qa_predictions(dataset, model, tokenizer):
    def predict_batch(batch):
        preds = [
            predict_qa(q, c, model, tokenizer)
            for q, c in zip(batch["question"], batch["context"])
        ]
        return {
            "predicted_answer": [p["best_answer"] for p in preds],
            "predicted_answer_start": [p["answer_start"] for p in preds],
            "predicted_answer_end": [p["answer_end"] for p in preds],
            "no_answer_score": [p["no_answer_score"] for p in preds],
        }
    return dataset.map(predict_batch, batched=True, batch_size=8)

### Metrics

In [ ]:
squad_metric = evaluate.load("squad_v2")


def answerability_metrics(predicted_ds, threshold, indices=None):
    if indices is None:
        indices = range(len(predicted_ds))
    y_true, y_pred = [], []
    for i in indices:
        ex = predicted_ds[i]
        y_true.append(int(ex["answerable"]))
        y_pred.append(1 if ex["no_answer_score"] <= threshold else 0)

    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], average=None, zero_division=0
    )
    return {
        "n": len(y_true),
        "n_unans": sum(1 for y in y_true if y == 0),
        "ans_P": p[1], "ans_R": r[1], "ans_F1": f[1],
        "unans_P": p[0], "unans_R": r[0], "unans_F1": f[0],
        "macro_F1": (f[0] + f[1]) / 2,
    }


def make_squad_inputs(dataset, threshold):
    preds, refs = [], []
    for i, ex in enumerate(dataset):
        ans = ex["predicted_answer"]
        if ex["no_answer_score"] > threshold:
            ans = ""
        preds.append({"id": str(i), "prediction_text": ans,
                      "no_answer_probability": 0.0})
        refs.append({
            "id": str(i),
            "answers": {
                "text": [ex["answer"]] if ex["answerable"] else [],
                "answer_start": [ex["answer_start"]] if ex["answerable"] else [],
            },
        })
    return preds, refs


def tune_no_answer_threshold(predicted_ds, n_thresholds=101):
    scores = np.asarray(predicted_ds["no_answer_score"], dtype=float)
    finite = scores[np.isfinite(scores)]
    if len(finite) == 0:
        raise ValueError("No finite no-answer scores.")

    thresholds = np.linspace(finite.min() - 1e-6, finite.max() + 1e-6, n_thresholds)
    best_t, best_f1, best_res = None, -1, None
    for t in thresholds:
        preds, refs = make_squad_inputs(predicted_ds, t)
        res = squad_metric.compute(predictions=preds, references=refs)
        if res["f1"] > best_f1:
            best_f1, best_t, best_res = res["f1"], float(t), res
    return best_t, best_res


def squad_v2_eval(predicted_ds, threshold, indices=None):
    ds = predicted_ds if indices is None else predicted_ds.select(list(indices))
    preds, refs = make_squad_inputs(ds, threshold)
    return squad_metric.compute(predictions=preds, references=refs)


def lang_indices(predicted_ds, lang):
    return [i for i, l in enumerate(predicted_ds["lang"]) if l == lang]

### Analysis of hypothesis

In [ ]:
def paired_answerability_analysis(original_val_ds, w40_preds, ext_preds, w40_thr, ext_thr):
    records = []
    for i in range(len(original_val_ds)):
        gold = int(original_val_ds[i]["answerable"])
        w40_pred = 1 if w40_preds[i]["no_answer_score"] <= w40_thr else 0
        ext_pred = 1 if ext_preds[i]["no_answer_score"] <= ext_thr else 0
        w40_ok = (gold == w40_pred)
        ext_ok = (gold == ext_pred)
        if w40_ok and ext_ok: cat = "both_correct"
        elif w40_ok and not ext_ok: cat = "w40_only_correct"
        elif ext_ok and not w40_ok: cat = "ext_only_correct"
        else: cat = "both_wrong"
        records.append({
            "lang": original_val_ds[i]["lang"], "gold": gold,
            "w40_pred": w40_pred, "ext_pred": ext_pred,
            "w40_correct": w40_ok, "ext_correct": ext_ok,
            "category": cat,
        })
    return pd.DataFrame(records)


def summarise_paired(df):
    cat_table = (
        df.groupby(["lang", "category"]).size()
          .unstack(fill_value=0)
          .reindex(columns=["both_correct", "w40_only_correct", "ext_only_correct", "both_wrong"], fill_value=0)
    )
    h3 = (
        df[df.category == "w40_only_correct"]
          .groupby(["lang", "gold"]).size()
          .unstack(fill_value=0)
          .rename(columns={0: "gold_unans", 1: "gold_ans"})
    )
    return cat_table, h3

### Combine and run

#### Load dataset and translate

In [ ]:
original_ds = get_dataset()
translated_full = build_translated(original_ds)
translated_ds = swap_to_translated(translated_full)

print(f"\nOriginal val: {len(original_ds['validation'])}")
print(f"Translated val: {len(translated_ds['validation'])}")


In [ ]:
print(original_ds)
print(translated_ds)

#### Fine-tune and load W40 model

In [ ]:
if not os.path.exists(EXT_DIR):
    ext_tokenizer, ext_model = fine_tune_qa_model(
        train_ds=translated_ds["train"],
        val_ds=translated_ds["validation"],
        model_name="roberta-base",
        output_dir=EXT_DIR,
        num_epochs=3, batch_size=8, lr=2e-5, seed=SEED,
    )
else:
    print(f"Loading extension from {EXT_DIR}")
    ext_tokenizer = AutoTokenizer.from_pretrained(EXT_DIR)
    ext_model = AutoModelForQuestionAnswering.from_pretrained(EXT_DIR).to(device)

print(f"Loading W40 from {W40_DIR}")
w40_tokenizer = AutoTokenizer.from_pretrained(W40_DIR)
w40_model = AutoModelForQuestionAnswering.from_pretrained(W40_DIR).to(device)

#### Generate predictions and threshold tuning

In [ ]:
print("Predict W40 on regular dataset...")
w40_val = add_qa_predictions(original_ds["validation"], w40_model, w40_tokenizer)

print("Predict extension on trans dataset...")
ext_val = add_qa_predictions(translated_ds["validation"], ext_model, ext_tokenizer)

w40_thr, w40_res = tune_no_answer_threshold(w40_val)
ext_thr, ext_res = tune_no_answer_threshold(ext_val)
print(f"W40 threshold: {w40_thr:.3f}, val SQuAD-v2 F1 = {w40_res['f1']:.3f}")
print(f"Ext threshold: {ext_thr:.3f}, val SQuAD-v2 F1 = {ext_res['f1']:.3f}")

#### Answerability

In [ ]:
rows = []
for name, preds, thr in [("W40 (orig)", w40_val, w40_thr),
                         ("Ext (trans)", ext_val, ext_thr)]:
    for lang in ["__overall__", "ar", "ko", "te"]:
        idxs = None if lang == "__overall__" else lang_indices(preds, lang)
        m = answerability_metrics(preds, thr, idxs)
        m.update({"model": name, "lang": lang})
        rows.append(m)
ans_df = pd.DataFrame(rows)[
    ["model", "lang", "n", "n_unans", "ans_P", "ans_R", "ans_F1",
     "unans_P", "unans_R", "unans_F1", "macro_F1"]
]
print(ans_df.to_string(index=False, float_format="%.3f"))

#### SQuAD-v2

In [ ]:
squad_rows = []
for name, preds, thr in [("W40 (orig)", w40_val, w40_thr),
                         ("Ext (trans)", ext_val, ext_thr)]:
    for lang in ["__overall__", "ar", "ko", "te"]:
        idxs = None if lang == "__overall__" else lang_indices(preds, lang)
        r = squad_v2_eval(preds, thr, idxs)
        squad_rows.append({"model": name, "lang": lang,
                           "EM": r["exact"], "F1": r["f1"]})
squad_df = pd.DataFrame(squad_rows)
print(squad_df.to_string(index=False, float_format="%.3f"))

### Hypothesis comparison

In [ ]:
paired_df = paired_answerability_analysis(
    original_ds["validation"], w40_val, ext_val, w40_thr, ext_thr
)
cat_table, h3_table = summarise_paired(paired_df)
print("Per-language decisions:")
print(cat_table.to_string())
print("\nH3-specific: among 'W40-only correct', gold answerability:")
print(h3_table.to_string())

#### LaTeX generation

In [ ]:
print("% Answerability")
print(ans_df.to_latex(index=False, float_format="%.3f"))
print("% SQuAD-v2")
print(squad_df.to_latex(index=False, float_format="%.3f"))
print("% Paired decisions")
print(cat_table.to_latex(float_format="%d"))

# Challenge Set Evaluation

## Challenge set preparation

In [ ]:
import json
import pandas as pd

URL_PAVLOS = "https://en.wikipedia.org/wiki/Apollo_11"
CREATOR_PAVLOS = "Pavlos Constantinou"

pairs_pavlos = [
    ("Kennedy Space Center",
     "Launched atop a Saturn V rocket from Kennedy Space Center in Florida on July 16 at 13:32 UTC.",
     "Πού εκτοξεύτηκε το Apollo 11;",
     "Πού περισυλλέχθηκε το Apollo 11;"),
    ("Michael Collins",
     "The mission was crewed by Commander Neil Armstrong, Command Module Pilot Michael Collins, and Lunar Module Pilot Edwin \"Buzz\" Aldrin, all of whom were on their second and final spaceflight.'",
     "Ποιος ήταν ο Πιλότος της Μονάδας Διοίκησης;",
     "Ποιος ήταν ο εφεδρικός Πιλότος της Μονάδας Διοίκησης;"),
    ("Eagle",
     "After a three-day transit, Armstrong and Aldrin descended to the surface aboard the LM Eagle, landing in the Sea of Tranquility (Mare Tranquillitatis) on July 20 at 20:17 UTC while Collins remained in lunar orbit aboard the CM Columbia.",
     "Ποιο σκάφος προσσεληνώθηκε;",
     "Ποιο σκάφος μετέφερε και τους τρεις αστροναύτες στην επιφάνεια της Σελήνης;"),
    ("Richard Nixon",
     "Together they spent around two and a half hours walking on the surface, planting an American flag, speaking with President Richard Nixon who was in the White House, deploying scientific instruments, and collecting 21.5 kg (47.5 lb) of lunar material.",
     "Ποιος πρόεδρος μίλησε με τους αστροναύτες;",
     "Ποιος αντιπρόεδρος μίλησε με τους αστροναύτες;"),
    ("3",
     "Crew size: 3. Members: Neil Armstrong; Michael Collins; Buzz Aldrin.",
     "Πόσα άτομα αποτελούσαν το πλήρωμα του Apollo 11;",
     "Πόσα άτομα αποτελούσαν το εφεδρικό πλήρωμα του Apollo 11;"),
]

URL_MAKS = "https://en.wikipedia.org/wiki/Lockheed_SR-71_Blackbird"
CREATOR_MAKS = "Maks Lokar"

ctx_sr71_1 = "The SR-71 was developed in the 1960s as a black project by Lockheed's Skunk Works division. American aerospace engineer Clarence 'Kelly' Johnson was responsible for many of the SR-71's innovative concepts.[2] Its shape was based on the Lockheed A-12, a pioneer in stealth technology with its reduced radar cross section, but the SR-71 was longer and heavier to carry more fuel and a crew of two in tandem cockpits. The SR-71 was revealed to the public in July 1964 and entered service in the United States Air Force (USAF) in January 1966.[3]"

ctx_sr71_2 = "During missions, the SR-71 operated at high speeds and altitudes (Mach 3.2 at 85,000 ft or 26,000 m), allowing it to evade or outrace threats.[4] If a surface-to-air missile launch was detected, the standard evasive action was to accelerate and outpace the missile.[5] Equipment for the plane's aerial reconnaissance missions included signals-intelligence sensors, side-looking airborne radar, and a camera.[4] On average, an SR-71 could fly just once per week because of the lengthy preparations needed. A total of 32 aircraft were built; 12 were lost in accidents, none to enemy action.[6][7]"

pairs_maks = [
    ("Lockheed A-12", 
     ctx_sr71_1, 
     "Na katerem letalu je temeljila oblika letala SR-71?", 
     "Katero letalo je temeljilo na obliki letala SR-71?"),
    ("Skunk Works", 
     ctx_sr71_1, 
     "Kateri oddelek družbe Lockheed je razvil letalo SR-71 kot »črni projekt«?", 
     "Kateri oddelek družbe Boeing je razvil letalo SR-71 kot »črni projekt«?"),
    ("two", 
     ctx_sr71_1, 
     "Koliko članov posadke se namesti v tandemski kabini letala SR-71?", 
     "Koliko članov posadke se namesti v tandemski kabini letala Lockheed A-12?"),
    ("accelerate and outpace the missile", 
     ctx_sr71_2, 
     "Kakšen je bil standardni manever izogibanja v primeru zaznave rakete zemlja-zrak?", 
     "Kakšen je bil standardni napadalni ukrep v primeru zaznave rakete zemlja-zrak?"),
    ("side-looking airborne radar", 
     ctx_sr71_2, 
     "Kakšna vrsta radarja je bila vključena v izvidniško opremo letala SR-71?", 
     "Kakšna vrsta sonarja je bila vključena v izvidniško opremo letala SR-71?")
]

records = []

# Pavlos' pairs
for i, (ans, ctx, q_ans, q_unans) in enumerate(pairs_pavlos, 1):
    assert ctx.count(ans) >= 1, f"Answer '{ans}' not found in Apollo context."
    start = ctx.index(ans)
    pid = f"pavlos_p{i:02d}"
    base = dict(pair_id=pid, created_by=CREATOR_PAVLOS, context=ctx, lang="el", source_url=URL_PAVLOS)
    records.append({"id": f"{pid}a", **base, "question": q_ans, "answerable": True, "answer_start": start, "answer": ans})
    records.append({"id": f"{pid}u", **base, "question": q_unans, "answerable": False, "answer_start": -1, "answer": ""})

# Maks' pairs
for i, (ans, ctx, q_ans, q_unans) in enumerate(pairs_maks, 1):
    assert ctx.count(ans) >= 1, f"Answer '{ans}' not found in SR-71 context."
    start = ctx.index(ans)
    pid = f"maks_p{i:02d}"
    base = dict(pair_id=pid, created_by=CREATOR_MAKS, context=ctx, lang="sl", source_url=URL_MAKS)
    records.append({"id": f"{pid}a", **base, "question": q_ans, "answerable": True, "answer_start": start, "answer": ans})
    records.append({"id": f"{pid}u", **base, "question": q_unans, "answerable": False, "answer_start": -1, "answer": ""})

# Write to json
with open("challenge.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"{len(records)} records written to challenge.json")


challenge_df = pd.DataFrame(records)
display(
    challenge_df[["id", "created_by", "lang", "question", "answerable", "answer", "answer_start"]]
)

## Week 36

### Setup

In [ ]:
import torch
import pandas as pd
import json
import re
from nltk.corpus import stopwords
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

# Freeze models
translation_model.eval()
for param in translation_model.parameters():
    param.requires_grad = False

W36_RULE_THRESHOLD = 0.3

LANG_CODES.update({
    "el": "ell_Grek",
    "sl": "slv_Latn"
})
print("Models and thresholds frozen.")

# Setup
english_stopwords = set(stopwords.words("english"))

def content_tokens(text):
    tokens = re.findall(r"\b[a-z]+\b", text.lower())
    return {token for token in tokens if token not in english_stopwords}

translation_cache = {}

def predict_rule(question, context, lang, threshold):
    if (lang, question) not in translation_cache:
        try:
            translated = translate_to_english(question, lang)
            translation_cache[(lang, question)] = translated
        except Exception:
            translation_cache[(lang, question)] = question
            
    t_q = translation_cache[(lang, question)]
    q_words = content_tokens(t_q)
    c_words = content_tokens(context)
    
    if not q_words:
        return False, t_q
        
    overlap = len(q_words & c_words) / len(q_words)
    return overlap >= threshold, t_q

### Run inference

In [ ]:
with open("challenge.json", "r", encoding="utf-8") as f:
    challenge_data = json.load(f)
df = pd.DataFrame(challenge_data)

# Majority class from training set
train_answerable = sum(ds["train"]["answerable"])
train_total = len(ds["train"])
majority_class = train_answerable >= (train_total / 2)

print(f"Training Majority Class: {'Answerable' if majority_class else 'Unanswerable'}")
print("Running Rule Baseline (Translating questions, please wait)...")

pred_majority = []
pred_rule = []
translated_questions = []

for _, row in df.iterrows():
    pred_majority.append(majority_class)
    
    p_rule, t_q = predict_rule(row["question"], row["context"], row["lang"], W36_RULE_THRESHOLD)
    pred_rule.append(p_rule)
    translated_questions.append(t_q)

df["pred_majority"] = pred_majority
df["pred_rule"] = pred_rule
df["translated_question"] = translated_questions

### Metrics

In [ ]:
def get_formal_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[False, True], zero_division=0
    )
    
    return {
        "Accuracy": acc,
        "Macro-F1": macro_f1,
        "Unans. P": p[0], "Unans. R": r[0], "Unans. F1": f1[0],
        "Ans. P": p[1], "Ans. R": r[1], "Ans. F1": f1[1]
    }

metric_rows = []
for lang in ["Overall", "el", "sl"]:
    subset = df if lang == "Overall" else df[df["lang"] == lang]
    if len(subset) == 0: continue
    
    y_true = subset["answerable"].astype(bool).tolist()
    
    m_metrics = get_formal_metrics(y_true, subset["pred_majority"].astype(bool).tolist())
    m_metrics.update({"Model": "Majority Class", "Language": lang, "N": len(subset)})
    metric_rows.append(m_metrics)
    
    r_metrics = get_formal_metrics(y_true, subset["pred_rule"].astype(bool).tolist())
    r_metrics.update({"Model": "Rule Baseline", "Language": lang, "N": len(subset)})
    metric_rows.append(r_metrics)

metrics_df = pd.DataFrame(metric_rows)
cols = ["Model", "Language", "N", "Accuracy", "Macro-F1", "Ans. P", "Ans. R", "Ans. F1", "Unans. P", "Unans. R", "Unans. F1"]
metrics_df = metrics_df[cols]

print("\nWeek 36 Formal Metrics")
display(metrics_df.round(3))
print("\nLaTeX Format:")
print(metrics_df.to_latex(index=False, float_format="%.3f"))


print("\n" + "="*80)
print("PER-PAIR INSPECTION (WEEK 36)")
print("="*80)

# Group by pair_id to print together
for pair_id, group in df.groupby("pair_id"):
    if len(group) != 2:
        continue
        
    ans_rec = group[group["answerable"] == True].iloc[0]
    unans_rec = group[group["answerable"] == False].iloc[0]
    
    print(f"\n[Pair: {pair_id}] (Lang: {ans_rec['lang']})")
    print(f" Gold Answerable Question: {ans_rec['question']}")
    print(f" Gold Unanswerable Question: {unans_rec['question']}")
    
    if ans_rec['lang'] in ['el', 'sl']:
        print(f" (Translated Ans: {ans_rec['translated_question']})")
        print(f" (Translated Unans: {unans_rec['translated_question']})")
    
    print(f" -> Majority Preds: Ans={ans_rec['pred_majority']} | Unans={unans_rec['pred_majority']}")
    print(f" -> Rule Preds: Ans={ans_rec['pred_rule']} | Unans={unans_rec['pred_rule']}")
    
    if ans_rec['pred_rule'] == False:
        print(" [!] Rule failed to answer the answerable question (False Negative)")
    if unans_rec['pred_rule'] == True:
        print(" [!] Rule failed to abstain on the unanswerable question (False Positive)")

## Week 37

### Setup

In [ ]:
import torch
import pandas as pd
import json
import re

print("Freezing Week 37 Model")

bio_model = QuestionConditioned().to(device)
bio_model.load_state_dict(torch.load("span_prediction_42.pt"))
bio_model.eval()

for param in bio_model.parameters():
    param.requires_grad = False
print("Week 37 BIO Model frozen.")


def calculate_token_f1(pred, gold):
    pred_tokens = pred.strip().split()
    gold_tokens = gold.strip().split()
    
    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0
        
    common = set(pred_tokens) & set(gold_tokens)
    num_same = len(common)
    if num_same == 0:
        return 0.0
        
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * (precision * recall) / (precision + recall)

### Run inference

In [ ]:
with open("challenge.json", "r", encoding="utf-8") as f:
    challenge_data = json.load(f)
df = pd.DataFrame(challenge_data)

print("Week 37 Span Predictions...")
pred_spans = []

for _, row in df.iterrows():
    p_span = predict_span(
        model=bio_model, 
        tokenizer=bio_tokenizer, 
        question=row["question"], 
        context=row["context"], 
        device=device
    )
    pred_spans.append(p_span)

df["pred_span"] = pred_spans
df["gold_span"] = df.apply(lambda x: x["answer"] if x["answerable"] else "", axis=1)

df["EM"] = (df["pred_span"].str.strip() == df["gold_span"].str.strip()).astype(int)
df["Token_F1"] = df.apply(lambda x: calculate_token_f1(x["pred_span"], x["gold_span"]), axis=1)

### Metrics

In [ ]:
def get_qa_metrics(subset):
    if len(subset) == 0: return {}
    return {
        "N": len(subset),
        "Exact Match (EM)": subset["EM"].mean(),
        "Token F1": subset["Token_F1"].mean(),
        "Abstention Acc": (subset["EM"] == 1).mean() if subset["gold_span"].str.len().eq(0).all() else None
    }

metric_rows = []
for lang in ["Overall", "el", "sl"]:
    subset = df if lang == "Overall" else df[df["lang"] == lang]
    if len(subset) == 0: continue
    
    metrics = get_qa_metrics(subset)
    metrics["Language"] = lang
    metric_rows.append(metrics)

metrics_df = pd.DataFrame(metric_rows)
cols = ["Language", "N", "Exact Match (EM)", "Token F1"]
metrics_df = metrics_df[cols]

print("\nWeek 37 Formal Metrics")
display(metrics_df.round(3))
print("\nLaTeX Format:")
print(metrics_df.to_latex(index=False, float_format="%.3f"))



print("PER-PAIR INSPECTION (WEEK 37)")

for pair_id, group in df.groupby("pair_id"):
    if len(group) != 2:
        continue
        
    ans_rec = group[group["answerable"] == True].iloc[0]
    unans_rec = group[group["answerable"] == False].iloc[0]
    
    print(f"\n[Pair: {pair_id}] (Lang: {ans_rec['lang']})")
    print(f" Gold Answerable Q: {ans_rec['question']}")
    print(f" Pred Answerable: '{ans_rec['pred_span']}' (Gold: '{ans_rec['gold_span']}')")
    print(f" -> EM: {ans_rec['EM']} | Token F1: {ans_rec['Token_F1']:.2f}")
    
    print(f" Gold Unanswerable Q: {unans_rec['question']}")
    print(f" Pred Unanswerable: '{unans_rec['pred_span']}' (Gold: '')")
    print(f" -> EM: {unans_rec['EM']} | Token F1: {unans_rec['Token_F1']:.2f}")
    
    # Highlight failures
    if ans_rec['pred_span'].strip() == "":
        print("Model falsely abstained on the answerable question.")
    elif ans_rec['EM'] == 0:
        print("Model predicted the wrong span for the answerable question.")
        
    if unans_rec['pred_span'].strip() != "":
        print("Model failed to abstain on the unanswerable question.")

## Week 39

### Setup

In [ ]:
import torch
import pandas as pd
import json
import joblib
import nltk
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from torch.nn.utils.rnn import pad_sequence

with open("challenge.json", "r", encoding="utf-8") as f:
    challenge_data = json.load(f)
df = pd.DataFrame(challenge_data)

def get_formal_metrics(y_true, y_pred):
    """Returns Accuracy, Macro-F1, and per-class P/R/F1."""
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    
    # P/R/F1 for [Unanswerable(False), Answerable(True)]
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[False, True], zero_division=0
    )
    
    return {
        "Accuracy": acc,
        "Macro-F1": macro_f1,
        "Unans. P": p[0], "Unans. R": r[0], "Unans. F1": f1[0],
        "Ans. P": p[1], "Ans. R": r[1], "Ans. F1": f1[1]
    }

def print_pair_inspection(df, pred_col, model_name, lang_filter=None):
    """Prints the paired challenge questions and model predictions."""
    print("\n" + "="*80)
    print(f"PER-PAIR INSPECTION: {model_name} " + (f"({lang_filter})" if lang_filter else "(All)"))
    print("="*80)
    
    subset = df if not lang_filter else df[df["lang"] == lang_filter]
    
    for pair_id, group in subset.groupby("pair_id"):
        if len(group) != 2: continue
        
        ans_rec = group[group["answerable"] == True].iloc[0]
        unans_rec = group[group["answerable"] == False].iloc[0]
        
        print(f"\n[Pair: {pair_id}] (Lang: {ans_rec['lang']})")
        print(f"  Answerable Q:   {ans_rec['question']}")
        print(f"  Prediction:     {ans_rec[pred_col]}")
        
        print(f"  Unanswerable Q: {unans_rec['question']}")
        print(f"  Prediction:     {unans_rec[pred_col]}")
        
        # Highlight failures
        if ans_rec[pred_col] == False:
            print("Model falsely abstained on the answerable question.")
        if unans_rec[pred_col] == True:
            print("Model failed to abstain on the unanswerable question.")

### _The rest_...

In [ ]:
print("Freezing Week 39 Logistic Regression")
lr_clf = joblib.load('week39_lr_model.pkl')
lr_vectorizer = joblib.load('week39_lr_vectorizer.pkl')

def extract_extra_features(questions, contexts):
    features = []
    for question, context in zip(questions, contexts):
        question_words = set(question.lower().split())
        context_words = set(context.lower().split())
        overlap_count = len(question_words & context_words)
        overlap_ratio = overlap_count / max(len(question_words), 1)
        question_length = len(question.split())
        context_length = len(context.split())
        length_ratio = question_length / max(context_length, 1)
        features.append([overlap_count, overlap_ratio, question_length, context_length, length_ratio])
    return np.array(features)

print("Running Logistic Regression predictions...")
lr_preds = []

for _, row in df.iterrows():
    q_vec = lr_vectorizer.transform([row["question"]])
    c_vec = lr_vectorizer.transform([row["context"]])
    extra = extract_extra_features([row["question"]], [row["context"]])
    
    X = hstack([q_vec, c_vec, csr_matrix(extra)]).tocsr()
    pred = lr_clf.predict(X)[0] # Returns 0 or 1
    lr_preds.append(bool(pred))

df["pred_lr"] = lr_preds

# Calculate LR Metrics
lr_metric_rows = []
for lang in ["Overall", "el", "sl"]:
    subset = df if lang == "Overall" else df[df["lang"] == lang]
    if len(subset) == 0: continue
    
    y_true = subset["answerable"].astype(bool).tolist()
    y_pred = subset["pred_lr"].astype(bool).tolist()
    
    metrics = get_formal_metrics(y_true, y_pred)
    metrics.update({"Model": "Logistic Regression", "Language": lang, "N": len(subset)})
    lr_metric_rows.append(metrics)

lr_metrics_df = pd.DataFrame(lr_metric_rows)
cols = ["Model", "Language", "N", "Accuracy", "Macro-F1", "Ans. P", "Ans. R", "Ans. F1", "Unans. P", "Unans. R", "Unans. F1"]
print("\nWeek 39 Logistic Regression: Formal Metrics")
display(lr_metrics_df[cols].round(3))

# Inspect
print_pair_inspection(df, "pred_lr", "Logistic Regression")

print("\nFreezing Week 39 LSTM Classifier")

# Load Vocab and Model
with open('week39_vocab.json', 'r') as f:
    week39_vocab = json.load(f)

lstm_model = AnswerabilityLSTM(len(week39_vocab)).to(device)
lstm_model.load_state_dict(torch.load('week39_lstm_model.pt'))
lstm_model.eval()
for param in lstm_model.parameters():
    param.requires_grad = False

print("Running LSTM predictions...")
lstm_preds = []

for _, row in df.iterrows():
    q_tokens = nltk.word_tokenize(row["question"])
    c_tokens = nltk.word_tokenize(row["context"])
    
    q_ids = [week39_vocab.get(t, week39_vocab["<UNK>"]) for t in q_tokens]
    c_ids = [week39_vocab.get(t, week39_vocab["<UNK>"]) for t in c_tokens]
    
    q_tensor = torch.tensor([q_ids], dtype=torch.long).to(device)
    c_tensor = torch.tensor([c_ids], dtype=torch.long).to(device)
    q_len = torch.tensor([len(q_ids)])
    c_len = torch.tensor([len(c_ids)])
    
    with torch.no_grad():
        logits = lstm_model(q_tensor, q_len, c_tensor, c_len)
        prob = torch.sigmoid(logits).item()
        pred = prob >= 0.5
        
    lstm_preds.append(pred)

df["pred_lstm"] = lstm_preds

# Calculate LSTM Metrics
lstm_metric_rows = []
for lang in ["Overall", "el", "sl"]:
    subset = df if lang == "Overall" else df[df["lang"] == lang]
    if len(subset) == 0: continue
    
    y_true = subset["answerable"].astype(bool).tolist()
    y_pred = subset["pred_lstm"].astype(bool).tolist()
    
    metrics = get_formal_metrics(y_true, y_pred)
    metrics.update({"Model": "LSTM Classifier", "Language": lang, "N": len(subset)})
    lstm_metric_rows.append(metrics)

lstm_metrics_df = pd.DataFrame(lstm_metric_rows)
print("\nWeek 39 LSTM Classifier: Formal Metrics")
display(lstm_metrics_df[cols].round(3))

print_pair_inspection(df, "pred_lstm", "LSTM Classifier")

# Print LaTeX
combined_metrics = pd.concat([lr_metrics_df, lstm_metrics_df], ignore_index=True)
print("\nCombined LaTeX Table")
print(combined_metrics[cols].to_latex(index=False, float_format="%.3f"))

## Week 40/41

### Challenge scores setup

In [ ]:
challenge_scores = pd.DataFrame(challenge_data)[
    ["id", "pair_id", "lang", "answerable"]
].copy()

def save_scores(score_table, run_df, name):
    saved = run_df[["id", "no_answer_score"]].rename(
        columns={"no_answer_score": f"{name}_score"}
    )
    score_table = score_table.drop(
        columns=[f"{name}_score"], errors="ignore"
    )
    return score_table.merge(
        saved, on="id", how="left", validate="one_to_one"
    )

### Setup

In [ ]:
import torch
import pandas as pd
import json
import evaluate
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

print("Freeze Week 40/41 Integrated QA Model")
model.eval()
for param in model.parameters():
    param.requires_grad = False

print(f"Threshold frozen at: {best_threshold}")


### Inference

In [ ]:
with open("challenge.json", "r", encoding="utf-8") as f:
    challenge_data = json.load(f)
df = pd.DataFrame(challenge_data)

print("Running Week 40/41 QA Predictions...")

pred_spans = []
pred_abstained = []
no_answer_scores = []

for _, row in df.iterrows():
    qa_preds = predict_qa(
        question=row["question"],
        context=row["context"],
        model=model,
        tokenizer=qa_tokenizer,
        max_length=384,
        stride=128,
        max_answer_length=30
    )
    
    score = qa_preds["no_answer_score"]
    is_abstained = score > best_threshold
    
    pred_spans.append("" if is_abstained else qa_preds["best_answer"])
    pred_abstained.append(is_abstained)
    no_answer_scores.append(score)

df["pred_span"] = pred_spans
df["pred_abstained"] = pred_abstained
df["no_answer_score"] = no_answer_scores
df["gold_span"] = df.apply(lambda x: x["answer"] if x["answerable"] else "", axis=1)

### Metrics

In [ ]:
challenge_scores = save_scores(challenge_scores, df, "base")

squad_metric = evaluate.load("squad_v2")

def get_qa_metrics(subset, lang_name):
    if len(subset) == 0: return {}
    
    # SQuAD-v2
    predictions = []
    references = []
    for idx, row in subset.iterrows():
        predictions.append({
            "id": str(idx),
            "prediction_text": row["pred_span"],
            "no_answer_probability": 1.0 if row["pred_abstained"] else 0.0
        })
        references.append({
            "id": str(idx),
            "answers": {
                "text": [row["gold_span"]] if row["answerable"] else [],
                "answer_start": [row["answer_start"]] if row["answerable"] else []
            }
        })
    squad_results = squad_metric.compute(predictions=predictions, references=references)
    
    # Answerability Metrics (Abstention vs Answer)
    y_true = subset["answerable"].astype(bool).tolist()
    y_pred = (~subset["pred_abstained"]).astype(bool).tolist()
    
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=[False, True], zero_division=0)
    
    return {
        "Language": lang_name,
        "N": len(subset),
        "SQuAD EM": squad_results["exact"],
        "SQuAD F1": squad_results["f1"],
        "Abstain Macro-F1": macro_f1,
        "Ans. P": p[1], "Ans. R": r[1], "Ans. F1": f1[1],
        "Unans. P": p[0], "Unans. R": r[0], "Unans. F1": f1[0]
    }

metric_rows = []
for lang in ["Overall", "el", "sl"]:
    subset = df if lang == "Overall" else df[df["lang"] == lang]
    metrics = get_qa_metrics(subset, lang)
    if metrics: metric_rows.append(metrics)

metrics_df = pd.DataFrame(metric_rows)
cols = ["Language", "N", "SQuAD EM", "SQuAD F1", "Abstain Macro-F1", "Ans. P", "Ans. R", "Ans. F1", "Unans. P", "Unans. R", "Unans. F1"]
metrics_df = metrics_df[cols]

print("\nWeek 40/41 Formal Metrics")
display(metrics_df.round(3))
print("\nLaTeX Format:")
print(metrics_df.to_latex(index=False, float_format="%.3f"))


print("PAIR INSPECTION (WEEK 40/41)")

for pair_id, group in df.groupby("pair_id"):
    if len(group) != 2:
        continue
        
    ans_rec = group[group["answerable"] == True].iloc[0]
    unans_rec = group[group["answerable"] == False].iloc[0]
    
    print(f"\n[Pair: {pair_id}] (Lang: {ans_rec['lang']})")
    
    # Answerable Question
    print(f"Gold Answerable Q: {ans_rec['question']}")
    print(f"Predicted Span: '{ans_rec['pred_span']}'")
    print(f"Gold Span: '{ans_rec['gold_span']}'")
    print(f"Abstained? {ans_rec['pred_abstained']} (Score: {ans_rec['no_answer_score']:.2f})")
    if ans_rec['pred_abstained']:
        print("Model falsely abstained on the answerable question.")
    elif ans_rec['pred_span'].strip() != ans_rec['gold_span'].strip():
        print("Model extracted the wrong span.")
        
    print()
        
    # Unanswerable Question
    print(f"Gold Unanswerable Q: {unans_rec['question']}")
    print(f"Predicted Span: '{unans_rec['pred_span']}'")
    print(f"Gold Span: ''")
    print(f"Abstained? {unans_rec['pred_abstained']} (Score: {unans_rec['no_answer_score']:.2f})")
    if not unans_rec['pred_abstained']:
        print("Model failed to abstain on the unanswerable question.")

## Extensions

### Shared evaluation code

In [ ]:
import torch
import pandas as pd
import json
import evaluate
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from scipy.stats import binomtest

with open("challenge.json", "r", encoding="utf-8") as f:
    challenge_data = json.load(f)
df = pd.DataFrame(challenge_data)

squad_metric = evaluate.load("squad_v2")

def print_pair_inspection(df, model_name):
    print(f"PAIR INSPECTION: {model_name}")
    
    for pair_id, group in df.groupby("pair_id"):
        if len(group) != 2: continue
        ans_rec = group[group["answerable"] == True].iloc[0]
        unans_rec = group[group["answerable"] == False].iloc[0]
        
        print(f"\n[Pair: {pair_id}] (Lang: {ans_rec['lang']})")
        print(f"Ans Q: {ans_rec['question']}")
        print(f"Pred: '{ans_rec['pred_span']}' (Abstained: {ans_rec['pred_abstained']})")
        print(f"Unans Q: {unans_rec['question']}")
        print(f"Pred: '{unans_rec['pred_span']}' (Abstained: {unans_rec['pred_abstained']})")
        
        if ans_rec['pred_abstained']: print("False Abstention")
        if not unans_rec['pred_abstained']: print("Failed Abstention")

### Pavlos'

#### Setup

In [ ]:
import torch
import pandas as pd
import json
import numpy as np
import evaluate
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

print("Freezing Week 40 Model and tuning thresholds")
model.eval()
for param in model.parameters():
    param.requires_grad = False

def normalize_answer(text):
    import string, re
    text = text.lower()
    text = "".join(c for c in text if c not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def exact_match(pred, gold):
    return int(normalize_answer(pred) == normalize_answer(gold))

def token_f1(pred, gold):
    from collections import Counter
    pred = normalize_answer(pred).split()
    gold = normalize_answer(gold).split()
    if not pred or not gold: return float(pred == gold)
    common = Counter(pred) & Counter(gold)
    same = sum(common.values())
    if same == 0: return 0.0
    precision = same / len(pred)
    recall = same / len(gold)
    return 2 * precision * recall / (precision + recall)

def evaluate_threshold(scores, threshold):
    y_true = np.array([s["gold_answerable"] for s in scores])
    y_pred = np.array([s["no_answer_score"] <= threshold for s in scores])
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    unanswerable = ~y_true
    unanswerable_recall = np.mean(~y_pred[unanswerable]) if unanswerable.any() else 0.0
    em, f1 = [], []
    for s, answerable in zip(scores, y_pred):
        pred = s["predicted_answer"] if answerable else ""
        gold = s["gold_answer"] if s["gold_answerable"] else ""
        em.append(exact_match(pred, gold))
        f1.append(token_f1(pred, gold))
    return {"threshold": float(threshold), "answerability_macro_f1": float(macro_f1), "unanswerable_recall": float(unanswerable_recall), "EM": float(np.mean(em)), "token_F1": float(np.mean(f1))}

#### Model preparation

In [ ]:
val_scores = []

for example in ds["validation"]:
    preds = predict_qa(example["question"], example["context"], model, qa_tokenizer)
    val_scores.append({
        "lang": example["lang"],
        "gold_answerable": bool(example["answerable"]),
        "gold_answer": example["answer"],
        "predicted_answer": preds["best_answer"],
        "no_answer_score": preds["no_answer_score"]
    })

# Global threshold
thresholds = np.concatenate(([-np.inf], np.unique([s["no_answer_score"] for s in val_scores]), [np.inf]))
results = [evaluate_threshold(val_scores, t) for t in thresholds]
baseline_result = max(results, key=lambda r: r["answerability_macro_f1"])
global_threshold = baseline_result["threshold"]

# Per-language thresholds
LANG_RECALL_TARGET = 0.85
per_lang_threshold = {}
for lang in ["ar", "ko", "te"]:
    lang_scores = [s for s in val_scores if s["lang"] == lang]
    lang_thresholds = np.concatenate(([-np.inf], np.unique([s["no_answer_score"] for s in lang_scores]), [np.inf]))
    lang_results = [evaluate_threshold(lang_scores, t) for t in lang_thresholds]
    max_recall = max(r["unanswerable_recall"] for r in lang_results)
    effective_target = min(LANG_RECALL_TARGET, max_recall - 1e-3)
    lang_eligible = [r for r in lang_results if r["unanswerable_recall"] >= effective_target]
    
    if lang_eligible:
        chosen = max(lang_eligible, key=lambda r: (r["answerability_macro_f1"], -abs(r["unanswerable_recall"] - effective_target)))
    else:
        chosen = max(lang_results, key=lambda r: r["answerability_macro_f1"])
    per_lang_threshold[lang] = chosen["threshold"]

print(f"Global Threshold: {global_threshold:.4f}")
print(f"Per-Language Thresholds: {per_lang_threshold}")

### Metrics and evaluation

In [ ]:
with open("challenge.json", "r", encoding="utf-8") as f:
    challenge_data = json.load(f)
df = pd.DataFrame(challenge_data)

print("\nRunning Extension on Challenge Set...")
pred_spans, pred_abstained, used_thresholds = [], [], []

for _, row in df.iterrows():
    qa_preds = predict_qa(
        question=row["question"], context=row["context"], model=model, tokenizer=qa_tokenizer
    )
    score = qa_preds["no_answer_score"]
    
    # Fallback to global threshold for unseen languages
    threshold = per_lang_threshold.get(row["lang"], global_threshold)
    
    is_abstained = score > threshold
    pred_spans.append("" if is_abstained else qa_preds["best_answer"])
    pred_abstained.append(is_abstained)
    used_thresholds.append(threshold)

df["pred_span"] = pred_spans
df["pred_abstained"] = pred_abstained
df["no_answer_score"] = [predict_qa(q, c, model, qa_tokenizer)["no_answer_score"] for q, c in zip(df["question"], df["context"])]
df["used_threshold"] = used_thresholds
df["gold_span"] = df.apply(lambda x: x["answer"] if x["answerable"] else "", axis=1)

squad_metric = evaluate.load("squad_v2")

def get_ext_metrics(subset, lang_name):
    if len(subset) == 0: return {}
    predictions = [{"id": str(idx), "prediction_text": r["pred_span"], "no_answer_probability": 1.0 if r["pred_abstained"] else 0.0} for idx, r in subset.iterrows()]
    references = [{"id": str(idx), "answers": {"text": [r["gold_span"]] if r["answerable"] else [], "answer_start": [r["answer_start"]] if r["answerable"] else []}} for idx, r in subset.iterrows()]
    squad_results = squad_metric.compute(predictions=predictions, references=references)
    
    y_true = subset["answerable"].astype(bool).tolist()
    y_pred = (~subset["pred_abstained"]).astype(bool).tolist()
    
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=[False, True], zero_division=0)
    
    return {
        "Language": lang_name, "N": len(subset),
        "SQuAD EM": squad_results["exact"], "SQuAD F1": squad_results["f1"],
        "Abstain Macro-F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Ans. P": p[1], "Ans. R": r[1], "Ans. F1": f1[1],
        "Unans. P": p[0], "Unans. R": r[0], "Unans. F1": f1[0]
    }

metric_rows = [get_ext_metrics(df if l == "Overall" else df[df["lang"] == l], l) for l in ["Overall", "el", "sl"]]
metrics_df = pd.DataFrame(metric_rows)
cols = ["Language", "N", "SQuAD EM", "SQuAD F1", "Abstain Macro-F1", "Ans. P", "Ans. R", "Ans. F1", "Unans. P", "Unans. R", "Unans. F1"]
metrics_df = metrics_df[cols]

print("\nFORMAL METRICS:")
display(metrics_df.round(3))
print("\nLaTeX Format:")
print(metrics_df.to_latex(index=False, float_format="%.3f"))

print("PAIR INSPECTION:")

for pair_id, group in df.groupby("pair_id"):
    if len(group) != 2: continue
    ans_rec, unans_rec = group[group["answerable"] == True].iloc[0], group[group["answerable"] == False].iloc[0]
    
    print(f"\n[Pair: {pair_id}] (Lang: {ans_rec['lang']})")
    print(f"Ans Q: {ans_rec['question']}")
    print(f"Pred: '{ans_rec['pred_span']}' (Score: {ans_rec['no_answer_score']:.2f}, Thresh: {ans_rec['used_threshold']:.2f}) -> Abstained: {ans_rec['pred_abstained']}")
    print(f"Unans Q: {unans_rec['question']}")
    print(f"Pred: '{unans_rec['pred_span']}' (Score: {unans_rec['no_answer_score']:.2f}, Thresh: {unans_rec['used_threshold']:.2f}) -> Abstained: {unans_rec['pred_abstained']}")
    
    if ans_rec['pred_abstained']: print("False Abstention")
    if not unans_rec['pred_abstained']: print("Failed Abstention")

challenge_scores = save_scores(challenge_scores, df, "calibrated")


## Maks'

#### Setup

In [ ]:
import torch
import pandas as pd
import json
import evaluate
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

print("Freezing Model")

# Greek and Slovene codes
LANG_CODES.update({
    "el": "ell_Grek",
    "sl": "slv_Latn"
})

ext_model.eval()
for param in ext_model.parameters():
    param.requires_grad = False
print(f"Extension model frozen. Using validation threshold: {ext_thr:.3f}")

with open("challenge.json", "r", encoding="utf-8") as f:
    challenge_data = json.load(f)
df = pd.DataFrame(challenge_data)

print("\nTranslating challenge questions to English...")
translated_questions = []

for _, row in df.iterrows():
    try:
        # Using the NLLB translation function from W36
        t_q = translate_to_english(row["question"], row["lang"])
    except Exception as e:
        print(f"Translation failed for {row['lang']}: {e}")
        t_q = row["question"] # fallback
    translated_questions.append(t_q)

df["translated_question"] = translated_questions

### Inference

In [ ]:
print("Running Extension QA Predictions...")
pred_spans = []
pred_abstained = []
no_answer_scores = []

for _, row in df.iterrows():
    # TRANSLATED QUESTIONS
    qa_preds = predict_qa(
        question=row["translated_question"], 
        context=row["context"],
        model=ext_model,
        tokenizer=ext_tokenizer,
        max_length=MAX_LENGTH,
        stride=STRIDE,
        max_answer_length=MAX_ANSWER_LEN
    )
    
    score = qa_preds["no_answer_score"]
    is_abstained = score > ext_thr
    
    pred_spans.append("" if is_abstained else qa_preds["best_answer"])
    pred_abstained.append(is_abstained)
    no_answer_scores.append(score)

df["pred_span"] = pred_spans
df["pred_abstained"] = pred_abstained
df["no_answer_score"] = no_answer_scores
df["gold_span"] = df.apply(lambda x: x["answer"] if x["answerable"] else "", axis=1)

### Metrics

In [ ]:
challenge_scores = save_scores(challenge_scores, df, "translated")

squad_metric = evaluate.load("squad_v2")

def get_ext_metrics(subset, lang_name):
    if len(subset) == 0: return {}
    
    predictions = [
        {"id": str(idx), "prediction_text": r["pred_span"], "no_answer_probability": 1.0 if r["pred_abstained"] else 0.0} 
        for idx, r in subset.iterrows()
    ]
    references = [
        {"id": str(idx), "answers": {"text": [r["gold_span"]] if r["answerable"] else [], "answer_start": [r["answer_start"]] if r["answerable"] else []}} 
        for idx, r in subset.iterrows()
    ]
    squad_results = squad_metric.compute(predictions=predictions, references=references)
    
    y_true = subset["answerable"].astype(bool).tolist()
    y_pred = (~subset["pred_abstained"]).astype(bool).tolist()
    
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=[False, True], zero_division=0)
    
    return {
        "Language": lang_name, "N": len(subset),
        "SQuAD EM": squad_results["exact"], "SQuAD F1": squad_results["f1"],
        "Abstain Macro-F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Ans. P": p[1], "Ans. R": r[1], "Ans. F1": f1[1],
        "Unans. P": p[0], "Unans. R": r[0], "Unans. F1": f1[0]
    }

metric_rows = [get_ext_metrics(df if l == "Overall" else df[df["lang"] == l], l) for l in ["Overall", "el", "sl"]]
metrics_df = pd.DataFrame(metric_rows)
cols = ["Language", "N", "SQuAD EM", "SQuAD F1", "Abstain Macro-F1", "Ans. P", "Ans. R", "Ans. F1", "Unans. P", "Unans. R", "Unans. F1"]
metrics_df = metrics_df[cols]

print("\nFormal Metrics")
display(metrics_df.round(3))
print("\nLaTeX Format:")
print(metrics_df.to_latex(index=False, float_format="%.3f"))

print("PAIR INSPECTION: ")


for pair_id, group in df.groupby("pair_id"):
    if len(group) != 2: continue
    ans_rec = group[group["answerable"] == True].iloc[0]
    unans_rec = group[group["answerable"] == False].iloc[0]
    
    print(f"\n[Pair: {pair_id}] (Lang: {ans_rec['lang']})")
    
    print(f"Gold Ans Q: {ans_rec['question']}")
    print(f"Translated: {ans_rec['translated_question']}")
    print(f"Pred: '{ans_rec['pred_span']}' (Score: {ans_rec['no_answer_score']:.2f}) -> Abstained: {ans_rec['pred_abstained']}")
    
    print(f"Gold Unans Q: {unans_rec['question']}")
    print(f"Translated: {unans_rec['translated_question']}")
    print(f"Pred: '{unans_rec['pred_span']}' (Score: {unans_rec['no_answer_score']:.2f}) -> Abstained: {unans_rec['pred_abstained']}")
    
    if ans_rec['pred_abstained']: print("False Abstention")
    if not unans_rec['pred_abstained']: print("Failed Abstention")

In [ ]:
score_columns = {
    "Base QA": "base_score",
    "Calibrated threshold": "calibrated_score",
    "Translated QA": "translated_score",
}

rows = []
for pair_id, group in challenge_scores.groupby("pair_id"):
    if len(group) != 2 or group["answerable"].nunique() != 2:
        raise ValueError(f"Pair {pair_id} must have one record of each class.")

    ans = group[group["answerable"].astype(bool)].iloc[0]
    unans = group[~group["answerable"].astype(bool)].iloc[0]

    for model, col in score_columns.items():
        delta = unans[col] - ans[col]  # positive = correct score ordering
        rows.append({
            "pair_id": pair_id,
            "lang": ans["lang"],
            "model": model,
            "signed_delta": delta,
            "absolute_delta": abs(delta),
        })

pair_differences = pd.DataFrame(rows)
summary = pair_differences.groupby("model").agg(
    pairs=("pair_id", "nunique"),
    mean_abs_delta=("absolute_delta", "mean"),
    median_abs_delta=("absolute_delta", "median"),
    mean_signed_delta=("signed_delta", "mean"),
    fraction_unanswerable_higher=("signed_delta", lambda x: (x > 0).mean()),
).reset_index()

print(pair_differences.round(3).to_string(index=False))
print(summary.round(3).to_string(index=False))

print(summary.to_latex(index=False, float_format="%.3f"))

In [ ]:
# Threshold used for each record, matching the evaluation logic.
challenge_scores["base_threshold"] = best_threshold

challenge_scores["calibrated_threshold"] = (
    challenge_scores["lang"]
    .map(per_lang_threshold)
    .fillna(global_threshold)
)

challenge_scores["translated_threshold"] = ext_thr

models = {
    "Base QA": ("base_score", "base_threshold"),
    "Calibrated threshold": ("calibrated_score", "calibrated_threshold"),
    "Translated QA": ("translated_score", "translated_threshold"),
}

rows = []
for model_name, (score_col, threshold_col) in models.items():
    for label, is_answerable in [
        ("Answerable", True),
        ("Unanswerable", False),
    ]:
        subset = challenge_scores[
            challenge_scores["answerable"].astype(bool) == is_answerable
        ]

        mean_score = subset[score_col].mean()
        mean_threshold = subset[threshold_col].mean()
        mean_margin = mean_score - mean_threshold

        rows.append({
            "model": model_name,
            "class": label,
            "n": len(subset),
            "mean_score": mean_score,
            "mean_threshold": mean_threshold,
            "signed_margin": mean_margin,
            "distance_of_means": abs(mean_margin),
            "fraction_above_threshold": (
                subset[score_col] > subset[threshold_col]
            ).mean(),
        })

threshold_summary = pd.DataFrame(rows)
print(threshold_summary.round(3).to_string(index=False))

print(threshold_summary.to_latex(index=False, float_format="%.3f", caption="Threshold Analysis for Each Model and Class"))